# Parfois product similarity: final notebook

**MADSAD case study (FEP, University of Porto).** For every product colourway, find at least 4 similar products and
explain why each one matches, so that commercial buyers can validate the result. This notebook also estimates the
sales and the price of a new product from its similar past products.

The notebook joins the three sprint notebooks, in order. Each part builds on the previous one and can be read on its
own. **Section numbers restart in each part**, so "section 4.5" means the one in the same part.

| Part | Sprint | What |
|---|---|---|
| 1 | Sprint 1 | Data cleaning and validation: from the raw CSVs to one clean row per colourway |
| 2 | Sprint 2 | Features and modelling: six feature blocks, one similarity score, weights chosen without a ground truth |
| 3 | Sprint 3 | Explanations of every match, sales and price estimates, improvements over the Sprint 2 baseline |

Every step is written here with pandas, NumPy, matplotlib, Pillow and LightGBM. Steps that
need large neural networks, the manual review or long runs were done once and their saved results are loaded as data.
The notebook writes no file. Run it from the repository root or from `notebooks/`.


In [ ]:
import hashlib
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_DIR, IMG_DIR, EMB_DIR = ROOT / "data" / "csv", ROOT / "data" / "images", ROOT / "data" / "embeddings"
KEY = "PROD_CLR_EQUIV"  # model + colour: the unit of analysis

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})


# Part 1: Sprint 1, data cleaning and validation

**Parfois product similarity, MADSAD case study (FEP, University of Porto).**

This part documents the preprocessing done in Sprint 1, from the two raw CSV files to a clean table with one row
per product colourway. Every automatic step is **written out here** with pandas, NumPy and Pillow, so it can be read
and re-run on its own. Two steps were not automatic and are **described, not redone**:

- the **AI image audit** (CLIP, about 1.5 h on a laptop CPU): its results were saved once and are only loaded here;
- the **manual review** by the five team members (photos, colours, categories): its decisions were saved in a change
  log, which is loaded and applied here.

| Step | Section |
|---|---|
| Load and normalise the raw CSVs | 1–2 |
| Profile missing values and duplicates | 3 |
| Collapse sizes (SKUs) into colourways, join sales | 4 |
| Link each colourway to its photo | 5 |
| Automatic quality checks | 6 |
| AI image audit (results only) | 7 |
| Manual review (described; change log applied) | 8 |
| Final cleaning | 9 |
| Result and checks | 10 |


## 1. The raw data

Two files from Parfois: `df_product.csv` has one row per **SKU** (a size of a product in one colour) and
`df_sales.csv` one row per **colourway**. We first read them exactly as they are.


In [ ]:
raw_prod = pd.read_csv(CSV_DIR / "df_product.csv", dtype=str, encoding="utf-8", low_memory=False)
raw_sales = pd.read_csv(CSV_DIR / "df_sales.csv", encoding="utf-8")
print(f"df_product: {raw_prod.shape[0]:,} rows x {raw_prod.shape[1]} columns")
print(f"df_sales:   {raw_sales.shape[0]:,} rows x {raw_sales.shape[1]} columns")
print(f"SKUs: {raw_prod['PROD_COD'].nunique():,} | colourways: {raw_prod[KEY].nunique():,} "
      f"| models (PROD_REF): {raw_prod['PROD_REF'].nunique():,}")
raw_prod[[KEY, "PROD_COD", "SZ_DES", "PROD_DES", "CLR_DES", "CAT_DES_EN", "GFA_DES_EN", "COMPOSITION",
          "PRICE_BASE_W_VAT", "PROG_IMAGE"]].head(6)

**The unit of analysis is the colourway** (`PROD_CLR_EQUIV`, e.g. `167718_BU`): the description, colour, category
and photo are the same for every size, and the brief asks for similar *products*, not sizes.

A first look at the text shows three problems: non-breaking spaces, padding, and materials separated by double spaces.


In [ ]:
text_cols = [c for c in raw_prod.columns if pd.api.types.is_string_dtype(raw_prod[c])]
nbsp = {c: int(raw_prod[c].str.contains("\xa0", regex=False).sum()) for c in text_cols}
print("non-breaking spaces (\\xa0):", {c: n for c, n in nbsp.items() if n})
pad = pd.Series({c: int((raw_prod[c].notna() & (raw_prod[c].str.strip() != raw_prod[c])).sum()) for c in text_cols})
pad = pad[pad > 0].sort_values(ascending=False)
print(f"leading/trailing spaces in {len(pad)} columns, the most affected:", pad.head(6).to_dict())
print("\nCOMPOSITION separates materials with double spaces:")
print(raw_prod["COMPOSITION"].dropna().head(3).map(repr).to_string())

## 2. Loading and normalising

- Codes are read as **strings**, so leading zeros survive (`CAT_COD` `"052"` must not become `52`).
- Every text column: `\xa0` → space, trimmed, repeated spaces collapsed.
- `COMPOSITION` becomes a canonical list: `"Zinc  Enamel"` and `"Enamel  Zinc"` both become `"Enamel; Zinc"`.


In [ ]:
STR_DTYPES = {c: str for c in ["PROD_REF", "PROD_REF_EQUIV", "BAR_COD", "SEA_COD", "CAT_COD", "GFA_COD", "GFS_COD",
                               "SUP_COD", "TARIFF_COD"]}

def normalise_text(s):
    return s.str.replace("\xa0", " ", regex=False).str.strip().str.replace(r"\s+", " ", regex=True)

prod = pd.read_csv(CSV_DIR / "df_product.csv", dtype=STR_DTYPES, encoding="utf-8", low_memory=False)
sales = pd.read_csv(CSV_DIR / "df_sales.csv", encoding="utf-8")
prod["COMPOSITION"] = prod["COMPOSITION"].str.strip().str.replace(r"\s{2,}", "; ", regex=True)
for col in prod.columns:
    if pd.api.types.is_string_dtype(prod[col]):
        prod[col] = normalise_text(prod[col])
prod["COMPOSITION"] = prod["COMPOSITION"].map(
    lambda s: "; ".join(sorted(set(s.split("; ")))) if isinstance(s, str) else s)
sales[KEY] = sales[KEY].str.strip()

before_after = pd.DataFrame({"raw": raw_prod["COMPOSITION"], "clean": prod["COMPOSITION"]}).dropna()
before_after[before_after["raw"] != before_after["clean"]].drop_duplicates().head(5)

## 3. Missing values and duplicates

Each column is labelled **empty** (≥ 99.9% missing), **constant** (one value), **sparse** (≥ 50% missing) or ok.


In [ ]:
n = len(prod)
miss = pd.DataFrame({"n_missing": prod.isna().sum(), "pct_missing": (prod.isna().mean() * 100).round(2),
                     "n_unique": prod.nunique()})
miss["status"] = "ok"
miss.loc[miss["pct_missing"] >= 50, "status"] = "sparse (>=50% missing)"
miss.loc[miss["n_unique"] <= 1, "status"] = "constant"
miss.loc[miss["n_missing"] >= n * 0.999, "status"] = "empty"
miss = miss.sort_values("pct_missing", ascending=False)
print(miss["status"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(9, 3.2))
pct = miss["pct_missing"].sort_values()
ax.barh(range(len(pct)), pct.to_numpy(), color=np.where(pct >= 50, "#c62828", "#1f5f8b"), height=1.0)
ax.set_yticks([])
ax.set_xlabel("% missing")
ax.set_title(f"Missing values in the {len(pct)} product columns (red: ≥ 50% missing)")
plt.show()

In [ ]:
prod_keys, sales_keys = set(prod[KEY]), set(sales[KEY])
pd.Series({
    "product: fully duplicated rows": prod.duplicated().sum(),
    "product: duplicated SKU code (PROD_COD)": prod["PROD_COD"].duplicated().sum(),
    "product: duplicated barcode (BAR_COD)": prod["BAR_COD"].duplicated().sum(),
    "product: duplicated PROD_COD_EQUIV (re-coded items, expected)": prod["PROD_COD_EQUIV"].duplicated().sum(),
    "sales: duplicated keys": sales[KEY].duplicated().sum(),
    "sales: keys not in the product table": len(sales_keys - prod_keys),
    "product: colourways without sales": len(prod_keys - sales_keys),
}, name="count").to_frame()

No real duplicates; re-coded items (`*_EQUIV`) repeat by design. Every sales row has a product, and 370 colourways
have no sales (new or never sold), so sales are joined with a **left join**.


## 4. From SKUs to colourways

- One row per colourway, with the list of sizes and a size-free description (`PROD_DES_BASE`:
  `"Earring ESSENTIALS Gold U"` → `"Earring ESSENTIALS Gold"`).
- Attributes used for similarity must be **the same for every size**. Where sizes disagree, the most common value is
  kept and the conflict recorded in `sku_conflicts` (flag `SKU_ATTR_CONFLICT`) for the review, instead of silently
  taking the first size.
- Sales are joined (left join).


In [ ]:
SKU_INVARIANT = ["CLR_COD", "CLR_DES", "CLR_TYPE", "CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CATEGORY_MATRIX",
                 "COMPOSITION", "MATERIAL", "FINISHING", "PRINT_TYPE", "OUTFIT", "DIMENSION", "NUMBER_OF_UNITS",
                 "THEME", "FASHIONTYPE", "PROD_SEG", "PRICE_BASE_W_VAT", "PROG_IMAGE"]

p = prod.copy()
p["PROD_DES_BASE"] = [d[: -len(sz) - 1] if isinstance(d, str) and isinstance(sz, str) and d.endswith(" " + sz) else d
                      for d, sz in zip(p["PROD_DES"], p["SZ_DES"])]
g = p.groupby(KEY, sort=True)
items = g.first()

varies = g[SKU_INVARIANT].nunique(dropna=False) > 1
items["sku_conflicts"] = varies.apply(lambda r: "|".join(r.index[r]), axis=1)
conflicted = varies.index[varies.any(axis=1)]
modes = p[p[KEY].isin(conflicted)].groupby(KEY)[SKU_INVARIANT].agg(lambda s: s.mode(dropna=False).iat[0])
items.loc[modes.index, SKU_INVARIANT] = modes

# a re-coded colourway owns SKUs of an old and a new model code: keep both
refs = pd.concat([p[[KEY, c]].set_axis([KEY, "ref"], axis=1) for c in ["PROD_REF", "PROD_REF_EQUIV"]])
items["refs_all"] = refs.dropna().drop_duplicates().groupby(KEY)["ref"].agg(lambda s: "|".join(sorted(s)))
items["n_skus"] = g.size()
items["sizes"] = g["SZ_DES"].agg(lambda s: "|".join(s.dropna().unique()))
items = items.reset_index().merge(sales, on=KEY, how="left")
items["issues"] = ""

print(f"{len(prod):,} SKUs -> {len(items):,} colourways (key unique: {items[KEY].is_unique})")
print(f"sales joined: {items['SALES_QTY'].notna().sum():,} of {len(sales):,} rows; "
      f"{items['SALES_QTY'].isna().sum()} colourways without sales")
items[[KEY, "PROD_DES_BASE", "sizes", "n_skus", "SALES_QTY"]].sample(5, random_state=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
items["n_skus"].value_counts().sort_index().plot.bar(ax=axes[0], color="#1f5f8b")
axes[0].set_title("Sizes (SKUs) per colourway")
axes[0].set_xlabel("")
conf = items["sku_conflicts"].str.split("|").explode().loc[lambda s: s != ""].value_counts()
conf.plot.barh(ax=axes[1], color="#e08a00")
axes[1].set_title(f"Attributes that differ between sizes ({(items['sku_conflicts'] != '').sum()} colourways)")
axes[1].set_ylabel("")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Linking each colourway to its photo

`PROG_IMAGE` looks like `/241/52/167718_BU_1`: season, category code, file name without extension. The photo is
linked through that **file stem**, never by building a file name from the product code (some names are irregular).

Repair: some paths drop the colour (`/242/52/212983_2`) while the file has it (`212983_HM_1.jpg`). An item without a
file is linked to an unused file named after it, only if that file is unambiguous (one candidate, one claimant).


In [ ]:
def add_issue(mask, code):
    items.loc[mask.fillna(False).astype(bool), "issues"] += code + ";"

parts = items["PROG_IMAGE"].str.extract(r"^/(?P<season>[^/]+)/(?P<cat>\d{2})/(?P<stem>[^/]+)$")
items["img_stem"] = parts["stem"].str.rstrip(".")
path_ok = parts["stem"].notna() & parts["season"].str.fullmatch(r"\d{3}").fillna(False)
add_issue(~path_ok & items["PROG_IMAGE"].notna(), "IMG_PATH_INVALID")

files = {os.path.splitext(f)[0]: f for f in os.listdir(IMG_DIR)}
items["img_file"] = items["img_stem"].map(files)

# repair: unused files that start with the item's code + "_"
orphans = [s for s in files if s not in set(items["img_stem"].dropna())]
claims = {}
for i, r in items[items["img_file"].isna()].iterrows():
    codes = {c for c in (r[KEY], r["PROD_CLR"]) if isinstance(c, str)}
    cands = [o for o in orphans if any(o.startswith(c + "_") for c in codes)]
    if len(cands) == 1:
        claims[i] = cands[0]
owners = pd.Series(list(claims.values()), dtype="object").value_counts()
repaired = pd.Series(False, index=items.index)
for i, stem in claims.items():
    if owners[stem] == 1:
        items.at[i, "img_file"], items.at[i, "img_stem"], repaired[i] = files[stem], stem, True

items["has_image"] = items["img_file"].notna()
add_issue(~items["has_image"], "IMG_FILE_MISSING")
add_issue(repaired, "IMG_PATH_REPAIRED")
used = set(items["img_file"].dropna())
print(f"image files: {len(files):,} | colourways with a photo: {items['has_image'].sum():,} of {len(items):,} "
      f"({items['has_image'].mean():.1%}) | repaired: {repaired.sum()} | files left unused: {len(set(files.values()) - used)}")

In [ ]:
def show(keys, title, table=None, n=6):
    rows = (items if table is None else table).set_index(KEY).loc[list(keys)[:n]]
    fig, axes = plt.subplots(1, len(rows), figsize=(2.3 * len(rows), 2.9))
    for ax, (k, r) in zip(np.atleast_1d(axes), rows.iterrows()):
        if isinstance(r.get("img_file"), str):
            ax.imshow(Image.open(IMG_DIR / r["img_file"]).convert("RGB"))
        ax.set_title(f"{k}\n{r['GFA_DES_EN']}\n{r['CLR_DES']}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

show(items.loc[items["has_image"], KEY].sample(6, random_state=3), "Typical product photos (packshots on white)")

The 18 files left unused are not broken links. Each belongs to a colourway whose sizes list **different photos**
(mostly re-coded items, with SKUs under an old and a new model code). A colourway uses the photo most of its sizes
list, so the others are not needed. All 18 colourways are flagged `SKU_ATTR_CONFLICT`, so the review looked at them.


## 6. Automatic quality checks

Each check adds an **issue code** with a severity. Nothing is changed or dropped here: the flags decide what the
manual review looks at first.

| Severity | Meaning |
|---|---|
| high | two sources contradict each other: must be reviewed |
| medium | probably a problem: should be reviewed |
| low | worth a look |
| info | a note (e.g. no photo, no sales): the item is kept and handled downstream |


### 6.1 Photo files: model, colour, readability, size, re-use


In [ ]:
# category folder in the path vs the item's category; file name of another model
add_issue(parts["cat"].notna() & (parts["cat"] != items["CAT_COD"]), "IMG_CAT_MISMATCH")
ref_ok = pd.Series([isinstance(s, str) and any(s.startswith(r) for r in refs.split("|"))
                    for s, refs in zip(items["img_stem"], items["refs_all"])], index=items.index)
add_issue(items["img_stem"].notna() & ~ref_ok, "IMG_REF_MISMATCH")

# colour code in the file name (<ref><colour>_<n>) vs the item's colour code; no colour = generic photo
img_clr = items["img_stem"].str.extract(r"^\d{6}([_\dA-Z]?[A-Z]{2})_")[0]
same = img_clr.str.lstrip("_") == items["CLR_COD"].str.lstrip("_")
add_issue(img_clr.notna() & ~same, "IMG_COLOUR_MISMATCH")
add_issue(items["img_stem"].notna() & img_clr.isna(), "IMG_GENERIC")

# open every photo once: content hash, readable, size and colour mode (~10 s)
md5, readable, shape = {}, {}, {}
for i, f in items["img_file"].dropna().items():
    path = IMG_DIR / f
    md5[i] = hashlib.md5(path.read_bytes()).hexdigest()
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            shape[i] = (im.width, im.height, im.mode)
        readable[i] = True
    except Exception:
        readable[i] = False
items["img_md5"] = pd.Series(md5, dtype="object")
add_issue(pd.Series(readable).reindex(items.index).eq(False), "IMG_UNREADABLE")
shape = pd.DataFrame.from_dict(shape, orient="index", columns=["img_width", "img_height", "img_mode"])
items[shape.columns] = shape
w, h = items["img_width"].astype(float), items["img_height"].astype(float)
add_issue(np.minimum(w, h) < 300, "IMG_SMALL")
add_issue((w / h > 1.5) | (h / w > 1.5), "IMG_ODD_SHAPE")
add_issue(items["img_mode"].notna() & (items["img_mode"] != "RGB"), "IMG_NOT_RGB")

# the same picture (same path or same bytes) used by another colour or model
img_key = items["img_md5"].fillna(items["PROG_IMAGE"])
n_clr = items.groupby(img_key)["CLR_COD"].transform("nunique")
n_ref = items.groupby(img_key)["PROD_REF"].transform("nunique")
add_issue(img_key.notna() & ((n_clr > 1) | (n_ref > 1)), "IMG_SHARED")

sizes = items.loc[items["has_image"]].groupby(["img_width", "img_height"]).size().sort_values(ascending=False)
print("most common photo sizes:", sizes.head(3).to_dict(), "| colour modes:", items["img_mode"].value_counts().to_dict())

### 6.2 Text against codes: colour and item type

- The key must be consistent: `PROD_CLR` = `PROD_REF` + `CLR_COD`.
- The letters of a colour code (`_BK`, `1BK` → `BK`) should always mean the same colour name.
- The description should mention the colour (`"Earring GLDN DEL Gold"`), and must not name *another* colour at its end.
- The first word of the description (the item type: *Necklace*, *Ring*...) is compared with the family. For each type
  word with at least 20 items, a family that covers under 5% of them is a contradiction (e.g. "Necklace ..." filed
  under Earrings), learned from the data itself.


In [ ]:
add_issue(items["PROD_CLR"] != items["PROD_REF"] + items["CLR_COD"], "TAB_KEY_INCONSISTENT")

letters = items["CLR_COD"].str.replace(r"^[_\d]", "", regex=True)
majority = items.groupby(letters)["CLR_DES"].agg(lambda s: s.mode().iat[0])
add_issue(items["CLR_DES"] != letters.map(majority), "CLR_CODE_NAME_INCONSISTENT")

colours = sorted(items["CLR_DES"].dropna().unique(), key=len, reverse=True)
def colour_in_desc(desc, clr):
    if not isinstance(desc, str) or not isinstance(clr, str):
        return "missing"
    if clr.lower() in desc.lower():
        return "ok"
    # colour words are Title Case; theme names are upper case ("SILVER BASICS"): match case-sensitively
    return "conflict" if any(desc.endswith(" " + c) for c in colours) else "not_found"
status = pd.Series([colour_in_desc(d, c) for d, c in zip(items["PROD_DES_BASE"], items["CLR_DES"])], index=items.index)
add_issue(status == "conflict", "CLR_CONFLICT_DESC")
add_issue(status == "not_found", "CLR_NOT_IN_DESC")

def singular(word):
    w = word.lower().strip()
    if len(w) <= 3 or not w.endswith("s") or w.endswith(("ss", "us", "is")):
        return w
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if w.endswith(("sses", "xes", "ches", "shes")):
        return w[:-2]
    return w[:-1]

def type_word(desc):
    if not isinstance(desc, str) or not desc:
        return None
    t = desc.split()
    if t[0].lower() == "set":  # "Set of 3 Rings ..." -> the type comes after
        t = t[2:] if len(t) > 1 and t[1].lower() == "of" else t[1:]
    return singular(t[0]) if t else None

tw = items["PROD_DES_BASE"].map(type_word)
counts = items.assign(tw=tw).groupby(["tw", "GFA_DES_EN"]).size()
totals = counts.groupby(level=0).transform("sum")
stats = pd.DataFrame({"share": counts / totals, "support": totals}).reset_index()
st = items.assign(tw=tw)[["tw", "GFA_DES_EN"]].merge(stats, how="left", on=["tw", "GFA_DES_EN"])
own_named = pd.Series([isinstance(d, str) and isinstance(f, str) and singular(f) in d.lower()
                       for d, f in zip(items["PROD_DES_BASE"], items["GFA_DES_EN"])])
add_issue(pd.Series(((st["support"] >= 20) & (st["share"] < 0.05) & ~own_named).to_numpy(), index=items.index),
          "TYPE_CONFLICT_DESC")
items.loc[items["issues"].str.contains("TYPE_CONFLICT_DESC"), [KEY, "PROD_DES_BASE", "GFA_DES_EN", "GFS_DES_EN"]]

### 6.3 Sales and prices

Flagged, never changed. The realised price is the sales amount divided by the units sold; amounts mix currencies,
so only extreme ratios (under 0.3× or over 1.5× the list price) are flagged.


In [ ]:
num = {c: pd.to_numeric(items[c], errors="coerce") for c in
       ["PRICE_BASE_W_VAT", "PRICE_OUTLET_W_VAT", "PRICE_LCP", "SALES_QTY", "SALES_AMT_FX_RATE"]}
base, qty, amt, cost = num["PRICE_BASE_W_VAT"], num["SALES_QTY"], num["SALES_AMT_FX_RATE"], num["PRICE_LCP"]
items["PRICE_BASE_W_VAT"] = base
add_issue(qty.isna(), "SALES_MISSING")
add_issue(qty == 0, "SALES_ZERO_QTY")
add_issue((qty < 0) | (amt < 0), "SALES_NEGATIVE")
add_issue((qty == 0) & (amt > 0), "SALES_AMT_WITHOUT_QTY")
add_issue(base.isna() | (base <= 0), "PRICE_INVALID")
add_issue(num["PRICE_OUTLET_W_VAT"] > base, "PRICE_OUTLET_ABOVE_BASE")
add_issue(cost.isna() | (cost <= 0), "PRICE_COST_MISSING")
add_issue(cost > base, "PRICE_BELOW_COST")
ratio = (amt / qty).where(qty > 0) / base
add_issue((ratio < 0.3) | (ratio > 1.5), "REALISED_PRICE_ODD")
add_issue(items["sku_conflicts"] != "", "SKU_ATTR_CONFLICT")

profile = items.assign(ratio=ratio).groupby("CAT_DES_EN").agg(
    colourways=(KEY, "size"), with_sales=("SALES_QTY", "count"), units_median=("SALES_QTY", "median"),
    units_p95=("SALES_QTY", lambda s: s.quantile(0.95)), units_max=("SALES_QTY", "max"),
    realised_vs_list=("ratio", "median"))
profile.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
for cat, grp in items.groupby("CAT_DES_EN"):
    ax.hist(np.log1p(grp["SALES_QTY"].dropna()), bins=40, histtype="step", linewidth=1.6, label=cat)
ax.set_xlabel("log(1 + units sold)")
ax.set_ylabel("colourways")
ax.set_title("Units sold per colourway, by category: not comparable across categories")
ax.legend(fontsize=8)
plt.show()

## 7. AI image audit (run once, results loaded here)

The rules above check file names, not what a photo shows. To find photos of the wrong product or colour, every photo
was processed **once**, on one laptop CPU (about 86 minutes for 9,457 photos), and the results were saved in
`data/embeddings/`. That step is not re-run here; this is what it did:

1. **Isolate the product**: crop the white background (pixel threshold); for photos that are not packshots (worn by
   a model, lifestyle), an object detector (OWLv2) finds the item.
2. **Encode the crop with CLIP** (ViT-B/32): 512 numbers per photo that describe what it shows. The same vectors are
   reused later as the visual feature for similarity.
3. **Zero-shot checks**: CLIP compares the photo with sentences such as *"a product photo of a necklace"* or
   *"... of a red item"*. A flag is raised when the item type or colour it sees disagrees with the data, when the
   photo is a near copy of another colour's photo, or when it is not a packshot.

Only the resulting flags are loaded:


In [ ]:
VIS_SEVERITY = {"VIS_TYPE_MISMATCH": "high", "VIS_COLOUR_MISMATCH": "medium", "IMG_NEAR_DUPLICATE": "medium",
                "VIS_TYPE_DOUBT": "low", "VIS_TARGET_NOT_FOUND": "low", "IMG_NOT_PACKSHOT": "low"}
audit = pd.read_parquet(ROOT / "data" / "embeddings" / "image_audit.parquet", columns=[KEY, "vis_issues", "mask_method"])
vis = items[KEY].map(audit.set_index(KEY)["vis_issues"]).fillna("")
for c in VIS_SEVERITY:
    add_issue(vis.str.contains(c, regex=False), c)
print(f"photos audited: {len(audit):,} | product isolated by:", audit["mask_method"].value_counts().to_dict())
audit["vis_issues"].str.split(";").explode().loc[lambda s: s != ""].value_counts().rename("photos").to_frame()

In [ ]:
cm = items[items["issues"].str.contains("VIS_COLOUR_MISMATCH")]
show(cm[KEY].sample(6, random_state=0), "VIS_COLOUR_MISMATCH: CLIP sees another colour than the data says")

### 6–7 together: what was flagged


In [ ]:
SEVERITY = {"IMG_PATH_INVALID": "high", "IMG_UNREADABLE": "high", "IMG_REF_MISMATCH": "high", "IMG_COLOUR_MISMATCH": "high",
            "TAB_KEY_INCONSISTENT": "high", "CLR_CODE_NAME_INCONSISTENT": "high", "CLR_CONFLICT_DESC": "high",
            "TYPE_CONFLICT_DESC": "high", "IMG_PATH_REPAIRED": "medium", "IMG_CAT_MISMATCH": "medium",
            "IMG_GENERIC": "medium", "IMG_SHARED": "medium", "SALES_NEGATIVE": "medium", "PRICE_INVALID": "medium",
            "SKU_ATTR_CONFLICT": "medium", "CLR_NOT_IN_DESC": "low", "SALES_ZERO_QTY": "low", "IMG_FILE_MISSING": "info",
            "SALES_MISSING": "info", "SALES_AMT_WITHOUT_QTY": "info", "REALISED_PRICE_ODD": "info",
            "PRICE_OUTLET_ABOVE_BASE": "info", "PRICE_COST_MISSING": "info", "PRICE_BELOW_COST": "info",
            "IMG_SMALL": "info", "IMG_ODD_SHAPE": "info", "IMG_NOT_RGB": "info", **VIS_SEVERITY}
RANK = {"high": 3, "medium": 2, "low": 1, "info": 0}
items["issues"] = items["issues"].str.rstrip(";")
items["priority"] = items["issues"].map(
    lambda s: max((RANK[SEVERITY[c]] for c in s.split(";") if c), default=-1)).map(
    {3: "high", 2: "medium", 1: "low", 0: "info", -1: "none"})
codes = items["issues"].str.split(";").explode().loc[lambda s: s != ""]
summary = codes.value_counts().rename("colourways").to_frame()
summary["severity"] = summary.index.map(SEVERITY)
print("priority of each colourway (its most severe flag):")
print(items["priority"].value_counts().reindex(["high", "medium", "low", "info", "none"]).to_string())

colour = {"high": "#c62828", "medium": "#e08a00", "low": "#6d4c9f", "info": "#9e9e9e"}
fig, ax = plt.subplots(figsize=(9, 6))
s = summary.iloc[::-1]
ax.barh(s.index, s["colourways"], color=s["severity"].map(colour))
ax.set_xscale("log")
ax.set_xlabel("colourways (log scale)")
ax.set_title("Issue codes found by the automatic checks")
for sev, col in colour.items():
    ax.bar(0, 0, color=col, label=sev)
ax.legend(title="severity", loc="lower right")
plt.show()

## 8. Manual review (described; its change log is applied)

The flags only point at possible problems; people decided. The 10,555 colourways were split into **five balanced
batches** (same category mix, all colours of a model in the same batch), one per team member, and reviewed in a
small browser app showing the photo, the description and the data side by side. For each colourway the reviewer chose:

| Decision | Meaning | What happens |
|---|---|---|
| Looks right | photo, description and data describe the same article | kept as is |
| Fix data | a value is wrong (colour, family, sub-family, finishing, material) | corrected; **only values that already exist in the data** could be chosen, so no new spelling enters |
| Wrong photo | the photo shows another product | photo unlinked; the item is matched on its data only |
| Not sure / Discard | decide together / remove | none were left at the end, no item was discarded |

The flagged items were reviewed first; in a second pass **every photo was checked**, flagged or not. The original
CSVs were never edited: every decision was saved, and the resulting changes are in `outputs/phase1/changes_log.csv`.


In [ ]:
log = pd.read_csv(ROOT / "outputs" / "phase1" / "changes_log.csv", encoding="utf-8-sig", keep_default_na=False)
print(f"{len(log)} changes on {log[KEY].nunique()} colourways:", log["column"].value_counts().to_dict())
log[log["column"] == "CLR_DES"].groupby(["old", "new"]).size().sort_values(ascending=False).head(8).rename("changes").to_frame()

In [ ]:
log[~log["column"].isin(["CLR_DES", "img_file"])][[KEY, "column", "old", "new"]]

Most fixes are colours that the data names too broadly (*Blue* for a turquoise item). Family fixes are items filed in
the wrong family (an ear cuff that is a ring). The decisions are applied to the table (the flags above stay as they were before the review: they
record why an item was looked at):


In [ ]:
reviewed = items.set_index(KEY)
for r in log.itertuples(index=False):
    if r.column == "img_file":           # wrong photo: unlinked
        reviewed.at[r.PROD_CLR_EQUIV, "img_file"] = None
    else:
        reviewed.at[r.PROD_CLR_EQUIV, r.column] = r.new
reviewed["has_image"] = reviewed["img_file"].notna()

# the decision per colourway (needed to know which photos a person accepted)
batches = pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig", keep_default_na=False, usecols=[KEY, "review_status"])
                     for f in sorted((ROOT / "outputs" / "phase1" / "batches").glob("batch_*_of_05.csv"))])
reviewed["review_status"] = batches.set_index(KEY)["review_status"].replace("", "ok")
reviewed = reviewed.reset_index()
print(reviewed["review_status"].value_counts().to_string())
print(f"photos after the review: {reviewed['has_image'].sum():,}")

## 9. Final cleaning

1. **Drop columns** that carry no information for similarity: empty or constant, exact copies of another column
   (Portuguese names, `*_ASIS`), and SKU-level columns (size, barcode) that mean nothing per colourway.
2. **Parse dates**, which come in three formats (`20230726`, `20230726.0`, ISO `2023-07-26 21:40:36`).
3. **Unify casing**: spellings that differ only in case get the most common one (`GOLDEN BASICS` → `Golden Basics`).
4. **Placeholders → missing**, per column only: `UNDEFINED` / `Not Applicable` (`DIMENSION`), `Without Block`,
   `Sem Categoria`. `Others` is kept: it is a real category in `PRINT_TYPE`, `GFA_DES_EN` and `FINISHING`.
5. **Sales features**: `has_sales`, `log_sales_qty`, `realised_price` (empty when 0 units) and
   `sales_pct_in_cat` (percentile **within the category**, since volumes differ so much between categories).
6. **Photo trust**: `img_trusted` (the photo shows this product) and `img_colour_trusted` (its colour is reliable
   too). A photo a reviewer accepted is trusted; otherwise the blocking flags decide.


In [ ]:
EMPTY_OR_CONSTANT = miss.index[miss["status"].isin(["empty", "constant"])].tolist()
COPIES = {"CAT_DES_PT": "CAT_DES_EN", "GFA_DES_PT": "GFA_DES_EN", "GFS_DES_PT": "GFS_DES_EN",
          "PRICE_BASE_W_VAT_ASWAS": "PRICE_BASE_W_VAT", "WEEKS_TARGET": "ITEM_LIFE_CYCLE_WEEK",
          "FASHIONTYPE_ASIS": "FASHIONTYPE", "THEME_ASIS": "THEME", "THEME_CODE_ASIS": "THEME_CODE",
          "PROD_TYPOLOGY_ORIG_ASIS": "PROD_TYPOLOGY_ORIG", "PURCHASE_BET_ORIG_ASIS": "PURCHASE_BET_ORIG",
          "ONL_ENTRY_DATE": "ONL_ENTRY_DATE_DT", "ONL_IMAGE_DATE": "ONL_IMAGE_DATE_DT"}
SKU_LEVEL = ["PROD_SPK", "PROD_COD", "PROD_COD_EQUIV", "BAR_COD", "SZ_COD", "SZ_DES", "PROD_DES", "DESCRIPTION"]
WORK = ["img_stem", "img_md5"]
clean = reviewed.drop(columns=[c for c in EMPTY_OR_CONSTANT + list(COPIES) + SKU_LEVEL + WORK if c in reviewed])
print(f"columns: {reviewed.shape[1]} -> {clean.shape[1]}")

# the copies hold the same values (the two date pairs: same dates, written 20231003.0 vs 2023-10-03)
def as_text(s):
    s = s.astype("string").str.replace(r"\.0$", "", regex=True)
    if s.name.startswith("ONL_"):
        s = pd.to_datetime(s, format="mixed", errors="coerce").astype("string")
    return s.fillna("")
same = {c: float((as_text(prod[c]) == as_text(prod[o])).mean()) for c, o in COPIES.items() if c in prod}
print("share of rows where the copy equals the original:", {c: round(v, 3) for c, v in same.items()})

In [ ]:
DATE_COLS = ["STORE_DATE_FINAL", "STORE_DATE_PLANNED", "STORE_DATE_REAL", "STORE_DATE_REVISED", "STORE_DATE_OUTLET_REAL",
             "START_DATE", "ONL_ENTRY_DATE_DT", "ONL_IMAGE_DATE_DT", "ECI_ONLINE_DATE", "AUDIT_DT_INSERT"]

def parse_dates(s):
    text = s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    compact = text.str.fullmatch(r"\d{8}").fillna(False)
    out = pd.to_datetime(text.where(compact), format="%Y%m%d", errors="coerce")
    return out.fillna(pd.to_datetime(text.where(~compact), format="ISO8601", errors="coerce"))

lost = {}
for c in [c for c in DATE_COLS if c in clean]:
    before = clean[c].notna().sum()
    clean[c] = parse_dates(clean[c])
    lost[c] = int(before - clean[c].notna().sum())
print("date values that could not be parsed:", lost)
clean["OPACITY"] = clean["OPACITY"].replace({"Transluncent": "Translucent"})  # one typo

In [ ]:
NO_RECASE = {KEY, "PROD_CLR", "PROD_REF", "PROD_REF_EQUIV", "PROD_DES_BASE", "PROG_IMAGE", "img_file", "issues",
             "sku_conflicts", "refs_all", "sizes", "review_status"}

def unify_casing(s):
    v = s.dropna()
    if v.empty or not v.map(type).eq(str).all():
        return s, 0
    canon = {}
    for value, _ in sorted(v.value_counts().items(), key=lambda kv: (-kv[1], kv[0])):
        canon.setdefault(value.casefold(), value)
    new = v.str.casefold().map(canon)
    changed = new != v
    out = s.copy()
    out.loc[new.index[changed]] = new[changed]
    return out, int(changed.sum())

recased = {}
for c in [c for c in clean if c not in NO_RECASE and pd.api.types.is_string_dtype(clean[c])]:
    clean[c], k = unify_casing(clean[c])
    if k:
        recased[c] = k
print("values re-cased:", recased)
theme = reviewed["THEME"].dropna()
theme.groupby(theme.str.casefold()).agg(lambda s: sorted(set(s))).loc[lambda s: s.map(len) > 1].head(5).to_frame()

In [ ]:
PLACEHOLDERS = {"DIMENSION": {"not applicable", "undefined"}, "DISTRIBUTION_BLOCK": {"without block"},
                "CATEGORY_MATRIX": {"sem categoria"}, "CATEGORY_MATRIX_BUYER": {"sem categoria"}}
for col, values in PLACEHOLDERS.items():
    is_ph = clean[col].astype("string").str.casefold().isin(values).fillna(False)
    clean.loc[is_ph, col] = None
    print(f"{col}: {is_ph.sum():,} placeholders set to missing")

qty = clean["SALES_QTY"]
clean["has_sales"] = qty.notna()
clean["log_sales_qty"] = np.log1p(qty)
clean["realised_price"] = (clean["SALES_AMT_FX_RATE"] / qty).where(qty > 0)
clean["sales_pct_in_cat"] = clean.groupby("CAT_DES_EN")["SALES_QTY"].rank(pct=True)

BLOCKING = {"IMG_UNREADABLE", "IMG_REF_MISMATCH", "VIS_TYPE_MISMATCH"}          # shows another product / broken
COLOUR_BLOCKING = BLOCKING | {"IMG_COLOUR_MISMATCH", "IMG_NEAR_DUPLICATE", "VIS_COLOUR_MISMATCH", "IMG_SHARED",
                              "IMG_GENERIC"}                                    # shape fine, colour unreliable
flags = clean["issues"].str.split(";").map(set)
accepted = clean["review_status"].isin({"ok", "fix"})
clean["img_trusted"] = clean["has_image"] & (accepted | flags.map(BLOCKING.isdisjoint))
clean["img_colour_trusted"] = clean["has_image"] & (accepted | flags.map(COLOUR_BLOCKING.isdisjoint))
clean = clean.sort_values(KEY).reset_index(drop=True)
print(f"photos trusted: {clean['img_trusted'].sum():,} | colour also trusted: {clean['img_colour_trusted'].sum():,}")

## 10. The result

One row per colourway, every colourway kept, ready for feature extraction (Sprint 2).


In [ ]:
assert clean[KEY].is_unique
assert len(clean) == raw_prod[KEY].nunique()
assert clean["SALES_QTY"].notna().sum() == len(raw_sales)
print(f"{clean.shape[0]:,} colourways x {clean.shape[1]} columns | checks passed: unique key, every colourway kept, "
      "every sales row joined")

overview = pd.DataFrame({
    "colourways": clean.groupby("CAT_DES_EN").size(),
    "families": clean.groupby("CAT_DES_EN")["GFA_DES_EN"].nunique(),
    "photo trusted": clean.groupby("CAT_DES_EN")["img_trusted"].mean().round(3),
    "with sales": clean.groupby("CAT_DES_EN")["has_sales"].mean().round(3),
    "median price €": clean.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].median(),
})
overview.loc["total"] = [len(clean), clean["GFA_DES_EN"].nunique(), round(clean["img_trusted"].mean(), 3),
                         round(clean["has_sales"].mean(), 3), clean["PRICE_BASE_W_VAT"].median()]
overview.astype({"colourways": int, "families": int})

In [ ]:
clean[[KEY, "PROD_DES_BASE", "CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CLR_DES", "FINISHING", "COMPOSITION",
       "PRICE_BASE_W_VAT", "sales_pct_in_cat", "img_trusted", "img_colour_trusted"]].sample(6, random_state=7)

## Summary

| | |
|---|---|
| Raw data | 17,125 SKUs (one per size) and 10,185 sales rows |
| Unit of analysis | **10,555 colourways**, every one kept |
| Text | non-breaking spaces and padding removed, `COMPOSITION` canonical, casing unified, placeholders → missing |
| Columns | empty, constant, copied and SKU-level columns dropped |
| Photos | 89.9% of colourways linked (33 paths repaired); file checks + AI audit on every photo; every photo checked by a person |
| Review | 461 colourways fixed (mostly colours), 122 wrong photos unlinked, none discarded; 588 logged changes |

Principles: the original files are never modified; nothing is dropped or relabelled silently (a flag, a human
decision and a change log); every step is deterministic, so everyone gets the same table.


# Part 2: Sprint 2, features and modelling

**Parfois product similarity, MADSAD case study (FEP, University of Porto).**

Sprint 1 produced a clean table with one row per colourway (`outputs/phase1/items_clean.csv`). This part does
the two halves of Sprint 2:

- **Features** (sections 1–10): each colourway becomes **numbers that can be compared**, one *block* of features per
  kind of information, so that the similarity of two products is computed block by block and can be explained
  ("same family", "same materials", "similar photo").
- **Modelling** (sections 11–16): the blocks are combined into **one similarity score**, the model is evaluated
  without a ground truth, and its weights are chosen first on automatic signals and then on the team's own judgement.

Every step that runs on a laptop is **written out here** with pandas, NumPy and Pillow. Three steps were not done in
this part; they are **described**, and their saved results are loaded:

- the **text embedding** of each product description (sentence model all-MiniLM-L6-v2), run once;
- the **image embedding** of each photo (CLIP ViT-B/32), run once;
- the team's **sorting of candidates** in a browser app (`outputs/phase4/feedback_*.jsonl`), used to learn the weights.

| Section | Block | How |
|---|---|---|
| 1 | — | Input and the common rule of every block |
| 2 | categorical | weighted one-hot of the product attributes |
| 3 | composition | materials, weighted by rarity |
| 4 | numeric | price within the category, launch date, season |
| 5 | — | filling missing material and metal, only where it is reliable |
| 6 | text | sentence embedding of a description (loaded) |
| 7 | image | CLIP embedding of the product crop (loaded) |
| 8 | palette | dominant colours of each photo (k-means in LAB) |
| 9 | — | colour names measured on the photos |
| 10 | — | the blocks side by side on one product |
| 11 | model | one score: the weighted mean of the block similarities, like-for-like candidates |
| 12 | evaluation | signals without a ground truth, tuning and test halves, random baseline |
| 13 | weights | step 1: guarded search on the signals; the photo-vs-attributes trade-off |
| 14 | weights | step 2: learned from the team's sorting (Bradley–Terry), blended 25% |
| 15 | model | the final model: top 10 for every colourway, results, ablations |
| 16 | — | the example of the brief, and why each match was chosen |
model about 30 times).


## 1. Input and the common rule

The input is the result of Sprint 1. Codes are read as text, so leading zeros survive.


In [ ]:
CODES = {c: str for c in ["PROD_REF", "PROD_REF_EQUIV", "SEA_COD", "CAT_COD", "GFA_COD", "GFS_COD"]}
items = pd.read_csv(ROOT / "outputs" / "phase1" / "items_clean.csv", encoding="utf-8-sig", dtype=CODES, low_memory=False)
items["STORE_DATE_FINAL"] = pd.to_datetime(items["STORE_DATE_FINAL"])
for c in ["img_trusted", "img_colour_trusted"]:
    items[c] = items[c].astype(str).eq("True")
items = items.sort_values(KEY, kind="stable").reset_index(drop=True)
print(f"{len(items):,} colourways x {items.shape[1]} columns | photos trusted: {items['img_trusted'].sum():,}")
items[[KEY, "CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CLR_DES", "FINISHING", "COMPOSITION", "PRICE_BASE_W_VAT"]].head()

**The rule every block follows.** Each product becomes a row of numbers of **length 1** (L2-normalised). The
similarity of two products in a block is then the dot product of their rows, the *cosine similarity*: 1 for
identical rows, 0 for nothing in common.

A block is also split into named **attributes** (groups of columns). The dot product of two rows is the sum of the
dot products of their attribute slices, so the score can be **explained**: e.g. 0.42 from the family, 0.20 from
the colour. A missing value is a row of zeros: it never counts as a match.


In [ ]:
def normalise_rows(m):
    norms = np.linalg.norm(m, axis=1, keepdims=True)
    return np.divide(m, norms, out=np.zeros_like(m), where=norms > 0).astype(np.float32)

def shared_attributes(m, attrs, i, j):
    """How much each attribute adds to the similarity of rows i and j (the parts add up to the score)."""
    out = {}
    for name, (start, stop) in attrs.items():
        part = float(m[i, start:stop] @ m[j, start:stop])
        if part > 1e-6:
            out[name] = round(part, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def neighbours(m, i, k=4, same_category=True):
    """Top k rows by cosine similarity: same category, other models only (another colour of the same model is
    trivially similar)."""
    s = m @ m[i]
    if same_category:
        s[(items["CAT_DES_EN"] != items["CAT_DES_EN"].iat[i]).to_numpy()] = -1
    s[(items["PROD_REF"] == items["PROD_REF"].iat[i]).to_numpy()] = -1
    top = np.argsort(-s, kind="stable")[:k]
    return top, s[top]

## 2. Categorical block

Each attribute is one-hot encoded, and each attribute gets a **weight** saying how much it matters for a buyer.
The value stored is √weight, so two products score the **sum of the weights they share** (before normalising).

Family (`GFA`) and sub-family (`GFS`) are encoded as **paths** (`Jewellery › Earrings › Hoop Earrings`), because names
such as "Others" repeat across categories and only mean something below their parent.

`FINISH_FAMILY` groups the ~30 finishing names into gold / rose gold / silver / black / other, so that "Golden" and
"Worn Gold" still get partial credit.


In [ ]:
FINISH_FAMILY = {
    "gold": ["Golden", "Worn Gold", "Brushed Golden", "Antik Golden", "Gold Plated", "Matte Golden", "Burnish Golden",
             "Light Golden", "Worn Yellow Gold", "Matt Brushed Gold", "Scratched Gold"],
    "rose gold": ["Rose Gold", "Brushed Rose Gold", "Worn Rose God"],
    "silver": ["Rodhium", "Silver", "Worn Rhodium", "Worn Silver", "Brushed Silver", "Antik Silver", "Brushed Rhodium",
               "Matte Silver", "Silver Plated"],
    "black": ["Black Nikel", "Black Plating"],
    "other": ["Other", "Multifinishing", "Cooper"],
}
FAMILY_OF = {v: fam for fam, vals in FINISH_FAMILY.items() for v in vals}
items["FINISH_FAMILY"] = items["FINISHING"].map(FAMILY_OF)

# attribute -> (source columns forming a path, weight)
CATEGORICAL = {
    "CAT": (["CAT_DES_EN"], 1.0),
    "GFA": (["CAT_DES_EN", "GFA_DES_EN"], 3.0),
    "GFS": (["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN"], 3.0),
    "CATEGORY_MATRIX": (["CATEGORY_MATRIX"], 1.5),
    "CLR_DES": (["CLR_DES"], 2.0),
    "CLR_TYPE": (["CLR_TYPE"], 0.5),
    "FINISHING": (["FINISHING"], 1.5),
    "FINISH_FAMILY": (["FINISH_FAMILY"], 1.0),
    "MATERIAL": (["MATERIAL"], 1.0),
    "PRINT_TYPE": (["PRINT_TYPE"], 1.0),
    "DIMENSION": (["DIMENSION"], 1.0),
    "OUTFIT": (["OUTFIT"], 1.0),
    "THEME": (["THEME"], 1.5),
    "FASHIONTYPE": (["FASHIONTYPE"], 0.5),
    "PROD_SEG": (["PROD_SEG"], 0.5),
    "NUMBER_OF_UNITS": (["NUMBER_OF_UNITS"], 0.5),
}

def categorical_block(df, spec=CATEGORICAL):
    parts, attrs, start = [], {}, 0
    for name, (cols, weight) in spec.items():
        values = df[cols[0]].astype("string")
        for c in cols[1:]:
            values = values + " › " + df[c].astype("string")       # missing if any part is missing
        levels = sorted(values.dropna().unique())
        codes = pd.Categorical(values, categories=levels).codes
        onehot = np.zeros((len(df), len(levels)), np.float32)
        onehot[np.flatnonzero(codes >= 0), codes[codes >= 0]] = np.sqrt(weight)
        parts.append(onehot)
        attrs[name] = (start, start + len(levels))
        start += len(levels)
    return normalise_rows(np.hstack(parts)), attrs

How filled and how varied each attribute is (before the missing values of section 5 are filled):


In [ ]:
summary = pd.DataFrame({name: {"columns": " › ".join(cols), "weight": w,
                               "values": items[cols].dropna().drop_duplicates().shape[0],
                               "filled": items[cols].notna().all(axis=1).mean()}
                        for name, (cols, w) in CATEGORICAL.items()}).T
summary.style.format({"filled": "{:.0%}"})

## 3. Composition block

`COMPOSITION` lists the materials (`"Brass; Glass; Zinc"`). Each material is one column, weighted by its **rarity**
(IDF, as in text search): sharing "Pearl" says more than sharing "Polyester", which half the apparel has.

IDF = log((1 + N) / (1 + products with the material)) + 1


In [ ]:
def composition_block(df):
    tokens = df["COMPOSITION"].fillna("").map(lambda s: sorted({t.strip() for t in str(s).split(";") if t.strip()}))
    vocab = sorted({t for ts in tokens for t in ts})
    col = {t: j for j, t in enumerate(vocab)}
    counts = pd.Series([t for ts in tokens for t in ts]).value_counts()
    idf = {t: float(np.log((1 + len(df)) / (1 + counts[t])) + 1) for t in vocab}
    m = np.zeros((len(df), len(vocab)), np.float32)
    for i, ts in enumerate(tokens):
        for t in ts:
            m[i, col[t]] = idf[t]
    return normalise_rows(m), {t: (j, j + 1) for t, j in col.items()}, counts, idf

comp, comp_attrs, counts, idf = composition_block(items)
print(f"{len(comp_attrs)} materials")
mat = pd.DataFrame({"products": counts, "weight (IDF)": pd.Series(idf).round(2)}).sort_values("products", ascending=False)
mat.iloc[list(range(6)) + list(range(-4, 0))]   # the most and the least common

## 4. Numeric block

- **Price**, compared **within the category** (a 30 € bag is cheap, a 30 € ring is not): its percentile in the
  category, 0 to 1.
- **Launch date** (`STORE_DATE_FINAL`): months since the first launch.
- **Season** (`SEA_COD`): one-hot.

A number cannot simply be one column in a cosine similarity, so price and date are encoded as **soft bands**:
Gaussian bumps at regular centres. Two products then score about exp(−Δ² / 4σ²): close values ≈ 1, far values ≈ 0.
Weights inside the block: price 2, launch 1, season 0.5.


In [ ]:
PRICE_SIGMA, LAUNCH_SIGMA = 0.10, 1.5          # in percentile points / in months
NUMERIC = {"price": 2.0, "launch": 1.0, "season": 0.5}

def soft_bands(x, centres, sigma):
    m = np.exp(-((x[:, None] - centres[None, :]) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    m[np.isnan(x)] = 0
    return normalise_rows(m)

def numeric_block(df):
    price_pct = df.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].rank(pct=True, method="average").to_numpy(float)
    months = ((df["STORE_DATE_FINAL"] - df["STORE_DATE_FINAL"].min()).dt.days / 30.44).to_numpy(float)
    # centres run 3σ past both ends, so values at the edges are compared like the others
    price_c = np.arange(-3 * PRICE_SIGMA, 1 + 3 * PRICE_SIGMA + 1e-9, PRICE_SIGMA / 2)
    launch_c = np.arange(np.floor(np.nanmin(months) - 3 * LAUNCH_SIGMA), np.ceil(np.nanmax(months) + 3 * LAUNCH_SIGMA) + 1e-9,
                         LAUNCH_SIGMA / 2)
    seasons = sorted(df["SEA_COD"].dropna().unique())
    codes = pd.Categorical(df["SEA_COD"], categories=seasons).codes
    season = np.zeros((len(df), len(seasons)), np.float32)
    season[np.flatnonzero(codes >= 0), codes[codes >= 0]] = 1
    parts = {"price": soft_bands(price_pct, price_c, PRICE_SIGMA), "launch": soft_bands(months, launch_c, LAUNCH_SIGMA),
             "season": season}
    attrs, start = {}, 0
    for name, m in parts.items():
        attrs[name] = (start, start + m.shape[1])
        start += m.shape[1]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(NUMERIC[n])) for n, m in parts.items()])), attrs

d = np.linspace(0, 0.4, 41)
centres = np.arange(-0.3, 1.3 + 1e-9, 0.05)
a = soft_bands(np.full(len(d), 0.5), centres, PRICE_SIGMA)
b = soft_bands(0.5 + d, centres, PRICE_SIGMA)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(d * 100, (a * b).sum(1), color="#1f5f8b", label="soft bands (computed)")
ax.plot(d * 100, np.exp(-d ** 2 / (4 * PRICE_SIGMA ** 2)), "--", color="#e08a00", label="exp(−Δ²/4σ²)")
ax.set_xlabel("difference in price percentile (points)")
ax.set_ylabel("price similarity")
ax.set_title("Two prices 10 points apart score ≈ 0.78, 30 apart ≈ 0.1")
ax.legend()
plt.show()

## 5. Filling missing material and metal, only where it is reliable

Two attributes buyers care about are often empty:

- `MATERIAL` (apparel only): the main fabric;
- the **metal** of jewellery (`FINISH_FAMILY`: gold, silver…).

They are filled **only for products with a trusted photo, and only when the guess is reliable**. The original values
are never replaced. Methods, in order:

1. **sibling** (material): another colour of the same model has a material (they agree 99.7% of the time);
2. **composition** (material): the composition is a single material that is also a `MATERIAL` value;
3. **neighbours**: a vote of the 15 most similar products of the category, weighted by similarity, kept when the
   winner has at least 80% of the votes. For the metal, the photo must agree: "gold" only if the colour palette is
   yellowish (mean b* ≥ 3), "silver" only if it is not (b* ≤ 6). Gold is 79% of the jewellery, so without this check
   silver items would too often be called gold.

Similarity for the vote: categorical block **without the attributes being guessed**, composition, CLIP image
embedding and colour palette (both loaded from `data/embeddings/`, see sections 7–8).

How reliable is each value? **5-fold cross-validation** on the products that do have it, with folds by model, so that
colours of the same model never help each other. A value is filled by the neighbours only if its precision is at least
90% on at least 10 predictions.


In [ ]:
def aligned(npy, meta_json):
    keys = json.loads((EMB_DIR / meta_json).read_text(encoding="utf-8"))["keys"]
    emb = np.load(EMB_DIR / npy).astype(np.float32)
    idx = items[KEY].map(pd.Series(range(len(keys)), index=keys))
    out = np.zeros((len(items), emb.shape[1]), np.float32)
    ok = idx.notna().to_numpy()
    out[ok] = emb[idx[ok].astype(int).to_numpy()]
    return out

image_raw = aligned("image_clip.npy", "image_clip.json")
palettes_saved = pd.read_parquet(EMB_DIR / "palettes.parquet").set_index(KEY)
palette_raw = np.stack([np.asarray(v, np.float32) if isinstance(v, (list, np.ndarray)) else np.zeros(24, np.float32)
                        for v in items[KEY].map(palettes_saved["palette_vector"])])
image_raw[~items["img_trusted"].to_numpy()] = 0
palette_raw[~items["img_trusted"].to_numpy()] = 0
yellow = items[KEY].map(pd.Series([sum(s * c[2] for s, c in zip(sh, lab)) for sh, lab in
                                   zip(palettes_saved["shares"], palettes_saved["lab"])], index=palettes_saved.index)).to_numpy()

VOTE_WEIGHTS = {"MATERIAL": {"categorical": 1.0, "composition": 3.0, "image": 1.0, "palette": 0.5},
                "FINISH_FAMILY": {"categorical": 1.0, "composition": 1.0, "image": 2.0, "palette": 2.0}}

def vote_features(target):
    spec = {k: v for k, v in CATEGORICAL.items() if k not in ("FINISHING", "MATERIAL", "FINISH_FAMILY")}
    parts = {"categorical": categorical_block(items, spec)[0], "composition": comp, "image": image_raw, "palette": palette_raw}
    w = VOTE_WEIGHTS[target]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(w[n])) for n, m in parts.items()]))

def knn_vote(Xq, Xt, yt, k=15):
    pred, conf = np.empty(len(Xq), dtype=object), np.zeros(len(Xq))
    for s in range(0, len(Xq), 512):
        sim = Xq[s:s + 512] @ Xt.T
        top = np.argsort(-sim, axis=1, kind="stable")[:, :k]
        for i, row in enumerate(top):
            votes = pd.Series(np.clip(sim[i, row], 0, None)).groupby(yt[row]).sum()
            if votes.sum() > 0:
                best = votes.sort_index().idxmax()
                pred[s + i], conf[s + i] = best, votes[best] / votes.sum()
    return pred, conf

def photo_agrees(pred, yel):
    pred, yel = np.asarray(pred, dtype=object), np.asarray(yel, float)
    return ~(((pred == "gold") & ~(yel >= 3.0)) | ((pred == "silver") & (yel > 6.0)))

In [ ]:
TARGETS = {"MATERIAL": "Apparel", "FINISH_FAMILY": "Jewellery"}
filled, evidence = [], {}
for col, cat in TARGETS.items():
    X = vote_features(col)
    in_cat = items["CAT_DES_EN"].eq(cat)
    lab = (in_cat & items[col].notna()).to_numpy()
    y, groups = items.loc[lab, col].to_numpy(), items.loc[lab, "PROD_REF"].to_numpy()

    # cross-validation, folds by model
    rng = np.random.default_rng(0)
    models = np.unique(groups)
    fold = pd.Series(rng.integers(0, 5, len(models)), index=models).loc[groups].to_numpy()
    pred, conf = np.empty(len(y), dtype=object), np.zeros(len(y))
    for f in range(5):
        pred[fold == f], conf[fold == f] = knn_vote(X[lab][fold == f], X[lab][fold != f], y[fold != f])
    ok_photo = photo_agrees(pred, yellow[lab]) if col == "FINISH_FAMILY" else np.ones(len(y), bool)
    kept = (conf >= 0.8) & ok_photo
    prec = pd.Series(pred[kept] == y[kept]).groupby(pred[kept]).agg(["mean", "size"])
    allowed = set(prec.index[(prec["mean"] >= 0.9) & (prec["size"] >= 10)])
    evidence[col] = {"products with the value": len(y), "accuracy (all)": (pred == y).mean(),
                     "share kept (vote ≥ 80%)": kept.mean(), "accuracy (kept)": (pred == y)[kept].mean(),
                     "values allowed": ", ".join(sorted(allowed))}

    # fill: trusted photo, value missing
    todo = in_cat & items[col].isna() & items["img_trusted"]
    guess = pd.Series(np.nan, index=items.index[todo], dtype=object)
    method = pd.Series("", index=guess.index)
    if col == "MATERIAL":
        vocab = set(items.loc[in_cat, col].dropna())
        sib = items[items[col].notna()].groupby("PROD_REF")[col].agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
        single = items["COMPOSITION"].fillna("").map(
            lambda s: [t.strip() for t in s.split(";") if t.strip()]).map(lambda t: t[0] if len(t) == 1 and t[0] in vocab else np.nan)
        for name, values in (("sibling", items["PROD_REF"].map(sib)), ("composition", single)):
            free = guess.isna() & values.loc[guess.index].notna()
            guess[free], method[free] = values.loc[guess.index[free]], name
    free = guess.index[guess.isna()]
    p, c = knn_vote(X[free.to_numpy()], X[lab], y)
    keep = (c >= 0.8) & np.isin(p, list(allowed))
    if col == "FINISH_FAMILY":
        keep &= photo_agrees(p, yellow[free.to_numpy()])
    guess[free[keep]], method[free[keep]] = p[keep], "neighbours"
    done = guess.notna()
    filled.append(pd.DataFrame({KEY: items.loc[guess.index[done], KEY], "column": col, "value": guess[done],
                                "method": method[done]}))
filled = pd.concat(filled, ignore_index=True)
pd.DataFrame(evidence).T

In [ ]:
print(filled.groupby(["column", "method"]).size().rename("values filled").to_string())
for col in TARGETS:
    sub = filled[filled["column"] == col].set_index(KEY)["value"]
    items[col] = items[col].where(items[col].notna(), items[KEY].map(sub))
print("\nfilled now:", {col: f"{items.loc[items['CAT_DES_EN'] == cat, col].notna().mean():.0%} of {cat}"
                        for col, cat in TARGETS.items()})

Not filled on purpose: the detailed finishing (golden vs worn gold vs brushed golden…) and the finishing of hand
bags, because the neighbours get them right too rarely.

With the filled values, the three tabular blocks are built:


In [ ]:
cat_m, cat_attrs = categorical_block(items)
num_m, num_attrs = numeric_block(items)
print(f"categorical {cat_m.shape} | composition {comp.shape} | numeric {num_m.shape}")

## 6. Text block (computed once, loaded here)

The categorical block only matches **identical** labels: "Golden Basics" and "GOLDEN DELICATES", or "Hoop Earrings" and
"Huggie", count as different. A sentence model knows they are close.

For each product a short English description is **built from the reviewed attributes** (never from the photo):


In [ ]:
def clean(v):
    return "" if pd.isna(v) else str(v).strip()

def describe(r):
    theme = clean(r["THEME"])
    theme = theme.title() if theme.isupper() else theme
    finish = f"Finish: {clean(r['FINISHING'])}. " if clean(r["FINISHING"]) else ""
    material = f"Made of {clean(r['COMPOSITION']).replace(';', ',')}. " if clean(r["COMPOSITION"]) else ""
    prt = f"Print: {clean(r['PRINT_TYPE'])}. " if clean(r["PRINT_TYPE"]) not in ("", "No Print") else ""
    return (f"{clean(r['GFS_DES_EN'])} ({clean(r['GFA_DES_EN'])}, {clean(r['CAT_DES_EN'])}). "
            f"Colour: {clean(r['CLR_DES']) or 'unknown'}. {finish}{material}{prt}Theme: {theme or 'none'}. "
            f"{clean(r['FASHIONTYPE'])}, {clean(r['PROD_SEG'])}.").strip()

text_meta = json.loads((EMB_DIR / "text.json").read_text(encoding="utf-8"))
for d in items.head(3).apply(describe, axis=1):
    print("•", d)

These sentences were encoded **once** with the sentence model `all-MiniLM-L6-v2` (about 1 minute on a CPU): each
becomes 384 numbers, the average of the model's token vectors, normalised to length 1. Products described in similar
words get close vectors. The result is loaded:


In [ ]:
text_m = normalise_rows(aligned("text.npy", "text.json"))
print(f"model: {text_meta['model']} | {text_meta['n']:,} products x {text_meta['dim']} numbers")
print("saved description of the first product:", text_meta["examples"][0])

## 7. Image block (computed once, loaded here)

The photos carry what the table cannot: shape, style, texture. In Sprint 1 every photo was processed once:

1. the **product is isolated**: the white background is cropped, and for photos with a model or a scene an object
   detector (OWLv2) finds the item, so a model's face or clothes do not drive the similarity;
2. the crop is encoded with **CLIP ViT-B/32**, a network trained on 400 million image–caption pairs: 512 numbers per
   photo, normalised to length 1.

Only photos the review **trusted** feed the block (`img_trusted`); the others are rows of zeros, so a wrong photo can
never make two products similar.


In [ ]:
image_m = normalise_rows(image_raw)
print(f"{(np.abs(image_m).sum(1) > 0).sum():,} trusted photos x {image_m.shape[1]} numbers")

def show(keys, title, scores=None):
    rows = items.set_index(KEY).loc[list(keys)]
    fig, axes = plt.subplots(1, len(rows), figsize=(2.3 * len(rows), 2.9))
    for n, (ax, (k, r)) in enumerate(zip(np.atleast_1d(axes), rows.iterrows())):
        if r["img_trusted"]:
            ax.imshow(Image.open(IMG_DIR / r["img_file"]).convert("RGB"))
        else:
            ax.text(0.5, 0.5, "no trusted photo", ha="center", color="#999")
        extra = "" if scores is None or n == 0 else f"  {scores[n - 1]:.2f}"
        ax.set_title(f"{k}{extra}\n{r['GFS_DES_EN']}\n{r['CLR_DES']}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

q = items.index[items[KEY] == "167718_BU"][0]            # the example of the brief
top, s = neighbours(image_m, q)
show([items[KEY].iat[q]] + items[KEY].iloc[top].tolist(), "Image block: the query and its 4 nearest photos", s)

## 8. Colour palette block

CLIP sees shape and style more than exact colours, and the colour *name* in the data is broad ("Blue" for navy and
turquoise alike). So each trusted photo also gets its **dominant colours**, measured on the pixels:

1. keep the **product pixels**: inside the product box found in Sprint 1, without the near-white background, resized
   to at most 96 px;
2. convert them to **CIELAB**, a colour space where distances match what the eye sees;
3. **k-means** with 4 clusters (k-means++ start, fixed seed): each cluster centre is a dominant colour, its size the
   colour's share; clusters under 5% are dropped;
4. **name** each colour after the nearest of 24 reference colours;
5. the **palette vector**: the shares summed per colour name, normalised. Two products both "mostly gold and white"
   get a high dot product.


In [ ]:
NAMED = {
    "black": (25, 25, 25), "white": (245, 245, 242), "grey": (128, 128, 128), "silver": (190, 190, 195),
    "gold": (205, 170, 90), "beige": (222, 203, 170), "camel": (190, 140, 85), "brown": (110, 70, 40),
    "red": (200, 35, 40), "burgundy": (115, 25, 45), "pink": (240, 150, 175), "fuchsia": (215, 45, 140),
    "orange": (240, 130, 45), "yellow": (245, 215, 55), "green": (45, 150, 75), "khaki": (120, 120, 65),
    "mint": (165, 220, 190), "teal": (10, 120, 120), "turquoise": (60, 195, 200), "light blue": (155, 195, 230),
    "blue": (45, 85, 190), "navy": (28, 38, 80), "purple": (110, 55, 150), "lilac": (190, 165, 220),
}
NAMES = list(NAMED)

def rgb_to_lab(rgb):
    c = np.asarray(rgb, dtype=np.float64) / 255.0
    c = np.where(c > 0.04045, ((c + 0.055) / 1.055) ** 2.4, c / 12.92)
    m = np.array([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.0721750], [0.0193339, 0.1191920, 0.9503041]])
    xyz = c @ m.T / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > (6 / 29) ** 3, np.cbrt(xyz), xyz / (3 * (6 / 29) ** 2) + 4 / 29)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]), 200 * (f[..., 1] - f[..., 2])], axis=-1)

NAMED_LAB = rgb_to_lab(np.array(list(NAMED.values())))

def kmeans(x, k, iters=20, seed=0):
    rng = np.random.default_rng(seed)
    k = min(k, len(np.unique(x, axis=0)))
    centres = [x[rng.integers(len(x))]]
    for _ in range(1, k):                                   # k-means++: spread the starting centres
        d = np.min(((x[:, None] - np.array(centres)[None]) ** 2).sum(-1), axis=1)
        centres.append(x[rng.choice(len(x), p=d / d.sum())])
    c = np.array(centres)
    for _ in range(iters):
        labels = ((x[:, None] - c[None]) ** 2).sum(-1).argmin(1)
        new = np.array([x[labels == j].mean(0) if (labels == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    return c, ((x[:, None] - c[None]) ** 2).sum(-1).argmin(1)

def product_pixels(img, box):
    img = img.convert("RGB")
    if box is not None and len(box) == 4:
        w, h = img.size
        x0, y0, x1, y1 = box
        img = img.crop((int(x0 * w), int(y0 * h), max(int(x0 * w) + 1, int(x1 * w)), max(int(y0 * h) + 1, int(y1 * h))))
    img.thumbnail((96, 96))
    a = np.asarray(img, dtype=np.uint8).reshape(-1, 3)
    keep = a.min(1) < 235                                    # near-white = background
    return a[keep] if keep.mean() > 0.02 else a              # an all-white product would vanish: keep the box

def palette(pixels, k=4):
    lab = rgb_to_lab(pixels)
    centres, labels = kmeans(lab, k)
    share = np.bincount(labels, minlength=len(centres)) / len(labels)
    keep = share >= 0.05
    centres, share = centres[keep], share[keep] / share[keep].sum()
    rgb = np.array([pixels[labels == j].mean(0) for j in np.flatnonzero(keep)])
    names = np.array(NAMES)[((centres[:, None, :] - NAMED_LAB[None]) ** 2).sum(-1).argmin(1)]
    order = np.argsort(-share, kind="stable")
    return [{"lab": centres[j], "rgb": rgb[j] / 255, "share": float(share[j]), "name": str(names[j])} for j in order]

def palette_vector(entries):
    v = np.zeros(len(NAMES), np.float32)
    for e in entries:
        v[NAMES.index(e["name"])] += e["share"]
    return v / np.linalg.norm(v) if np.linalg.norm(v) > 0 else v

In [ ]:
masks = pd.read_parquet(EMB_DIR / "masks.parquet").set_index(KEY)
todo = items[items["img_trusted"]]
pal, pal_vec = {}, np.zeros((len(items), len(NAMES)), np.float32)
for i, r in todo.iterrows():                                  # every trusted photo, about 2.5 minutes
    box = masks["box"].get(r[KEY])
    box = None if box is None or (isinstance(box, float) and np.isnan(box)) else list(box)
    with Image.open(IMG_DIR / r["img_file"]) as img:
        pal[r[KEY]] = palette(product_pixels(img, box))
    pal_vec[i] = palette_vector(pal[r[KEY]])
palette_m = normalise_rows(pal_vec)
print(f"palettes: {len(pal):,} photos | colours per photo:",
      pd.Series([len(p) for p in pal.values()]).value_counts().sort_index().to_dict())

In [ ]:
sample = todo[KEY].sample(8, random_state=1).tolist()
fig, axes = plt.subplots(2, len(sample), figsize=(2.1 * len(sample), 3.6), gridspec_kw={"height_ratios": [4, 1]})
for n, k in enumerate(sample):
    r = items.set_index(KEY).loc[k]
    axes[0, n].imshow(Image.open(IMG_DIR / r["img_file"]).convert("RGB"))
    axes[0, n].set_title(f"{r['GFS_DES_EN']}\n{r['CLR_DES']}", fontsize=8)
    x = 0
    for e in pal[k]:
        axes[1, n].barh(0, e["share"], left=x, color=e["rgb"])
        x += e["share"]
    axes[1, n].set_title(", ".join(dict.fromkeys(e["name"] for e in pal[k])), fontsize=7)
    for ax in axes[:, n]:
        ax.axis("off")
fig.suptitle("Photos and their dominant colours (width = share of the product pixels)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

**Metals look like light neutrals.** On a packshot the largest cluster of a gold item is its highlights: light and
barely saturated, the same as beige or ecru; silver looks white or grey. So the palette does not tell the metal; the
metal comes from `FINISHING`. Palette *similarity* still works, because two gold items get the same palette.


## 9. Colour names measured on the photos

The data has about 100 colour names, many of them shades of one colour ("Navy", "Midnight Blue", "Blue"). As a one-hot
they are as different as "Blue" and "Red". Here each name gets a **reference colour measured on the photos**: the median
LAB of the dominant palette colour of the packshots with that name (at least 5 photos). Two names are then compared by
their distance in LAB, with lightness counting half (a lighter shade of a colour is still that colour):
similarity = exp(−d² / 2τ²), τ = 8.

Multicolour and metallic names get no reference (one colour means nothing for a print, and gold measures as beige).
This table is used later to **explain** matches ("Navy ≈ Midnight Blue"); it is not a score block, because a soft
colour match in the categorical block changed none of the evaluation numbers.


In [ ]:
METALLIC = {"Gold", "Light Gold", "Old Gold", "Rose Gold", "Silver", "Old Silver", "Nickel", "Black Nickel", "Copper", "Bronze"}
packshot = masks["packshot"].reindex(items[KEY]).fillna(False).to_numpy(bool)
use = items["img_colour_trusted"].to_numpy() & packshot & items[KEY].isin(pal).to_numpy()
dom = np.stack([pal[k][0]["lab"] for k in items.loc[use, KEY]])     # the largest colour of each photo
ref = pd.DataFrame(dom, columns=["L", "a", "b"]).groupby(items.loc[use, "CLR_DES"].to_numpy()).median()
ref["photos"] = items.loc[use, "CLR_DES"].value_counts()
ref = ref[(ref["photos"] >= 5) & ~ref.index.str.lower().str.contains("multicolor") & ~ref.index.isin(METALLIC)]

lab = ref[["L", "a", "b"]].to_numpy() * np.array([0.5, 1, 1])
sim = np.exp(-((lab[:, None] - lab[None]) ** 2).sum(-1) / (2 * 8.0 ** 2))
names = ref.index.to_numpy()
close = {n: ", ".join(f"{names[j]} {sim[i, j]:.2f}" for j in np.argsort(-sim[i], kind="stable") if j != i and sim[i, j] >= 0.3)
         for i, n in enumerate(names)}
print(f"{len(ref)} colour names with a measured reference colour")
pd.DataFrame({"photos": ref["photos"], "closest names": pd.Series(close)}).loc[
    lambda d: d.index.isin(["Navy", "Blue", "Turquoise", "Beige", "Ecru", "Burgundy", "Khaki", "Pink"])]

In [ ]:
def lab_to_rgb(L, a, b):
    fy = (L + 16) / 116
    f = np.array([fy + a / 500, fy, fy - b / 200])
    xyz = np.where(f > 6 / 29, f ** 3, 3 * (6 / 29) ** 2 * (f - 4 / 29)) * np.array([0.95047, 1.0, 1.08883])
    m = np.array([[3.2404542, -1.5371385, -0.4985314], [-0.9692660, 1.8760108, 0.0415560], [0.0556434, -0.2040259, 1.0572252]])
    c = np.clip(m @ xyz, 0, 1)
    return np.where(c > 0.0031308, 1.055 * c ** (1 / 2.4) - 0.055, 12.92 * c)

top = ref.sort_values("photos", ascending=False).head(36)
chroma = np.hypot(top["a"], top["b"])
top = top.assign(order=np.where(chroma < 8, -1000 + top["L"], np.degrees(np.arctan2(top["b"], top["a"])) % 360)).sort_values("order")  # neutrals first, then by hue
fig, ax = plt.subplots(figsize=(11, 2.6))
for n, (name, r) in enumerate(top.iterrows()):
    ax.add_patch(plt.Rectangle((n, 0), 0.95, 1, color=lab_to_rgb(r.L, r.a, r.b)))
    ax.text(n + 0.47, -0.08, name, rotation=60, ha="right", va="top", fontsize=7)
ax.set_xlim(0, len(top))
ax.set_ylim(-0.1, 1)
ax.axis("off")
ax.set_title("Reference colour of the 36 most photographed colour names, measured on the photos")
plt.show()

## 10. The blocks side by side

Six blocks, one per kind of information, all with the same rules (rows of length 1, missing = zeros):


In [ ]:
blocks = {"categorical": cat_m, "composition": comp, "numeric": num_m, "text": text_m, "image": image_m, "palette": palette_m}
pd.DataFrame({name: {"numbers per product": m.shape[1], "products with the block": int((np.abs(m).sum(1) > 0).sum()),
                     "share": (np.abs(m).sum(1) > 0).mean()} for name, m in blocks.items()}).T.style.format({"share": "{:.1%}"})

Each block finds **different** neighbours for the same product, which is why they are combined. For the example of the
brief, the best match of each block, and *why* (the attributes behind the categorical score):


In [ ]:
rows = []
for name, m in blocks.items():
    top, s = neighbours(m, q, k=1)
    j = top[0]
    rows.append({"block": name, "best match": items[KEY].iat[j], "score": round(float(s[0]), 2),
                 "sub-family": items["GFS_DES_EN"].iat[j], "colour": items["CLR_DES"].iat[j],
                 "price": items["PRICE_BASE_W_VAT"].iat[j]})
r = items.iloc[q]
print(f"query {r[KEY]}: {r['GFS_DES_EN']}, {r['CLR_DES']}, {r['FINISHING']}, {r['PRICE_BASE_W_VAT']} €")
pd.DataFrame(rows).set_index("block")

In [ ]:
top, s = neighbours(cat_m, q, k=4)
for j, score in zip(top, s):
    print(f"{items[KEY].iat[j]}  score {score:.2f}  =", shared_attributes(cat_m, cat_attrs, q, j))

The categorical score of each neighbour is the sum of the attributes they share: this is how the final system will
**explain** a match. The next sections combine the six blocks into one score.


## 11. The model: one score from the six blocks

The similarity of a candidate *c* to a query *q* is a **weighted mean of the block similarities**:

score(q, c) = Σ_b w_b · sim_b(q, c) / Σ_b w_b, over the blocks *b* that **both** products have.

- sim_b is the dot product of the two rows of block *b* (section 1), so every part is between 0 and 1.
- A product without a trusted photo has no `image` or `palette` row. It is then scored on its other blocks, with
  the weights rescaled to sum to 1: a wrong photo can never make two products similar, and every product still gets
  matches. The `palette` block also needs a photo whose **colour** was checked in the review (`img_colour_trusted`).
- Each block's share of the score, w_b · sim_b / Σ w_b, is its **contribution**: the contributions add up to the
  score, and they are what the final system shows to explain a match ("62% from the photo, 25% from the family").

**Candidates.** Only products of the **same category** (`CAT_DES_EN`: a bag is never proposed for a ring), never the
product itself, and never another colour of the **same model** (`PROD_REF` or `PROD_REF_EQUIV`): those are trivially
similar and a buyer already knows them. Scores are rounded to 6 decimals and ties go to the lower key, so every
computer gets the same list.

The **weights** are what this sprint has to choose. The first ones were set by judgement: the photo and the
categorical attributes matter most, the four smaller blocks add detail.


In [ ]:
palette_m[~items["img_colour_trusted"].to_numpy()] = 0
blocks["palette"] = palette_m
BLOCKS = ["image", "categorical", "text", "composition", "palette", "numeric"]
W_JUDGEMENT = {"image": 0.3, "categorical": 0.3, "text": 0.1, "composition": 0.1, "palette": 0.1, "numeric": 0.1}

has = {b: (np.abs(m).sum(1) > 0).astype(np.float32) for b, m in blocks.items()}
MODEL_CODES = [items[c].astype("string").fillna("").to_numpy(object) for c in ["PROD_REF", "PROD_REF_EQUIV"]]
CATEGORY_POOLS = [np.asarray(ix) for _, ix in sorted(items.groupby("CAT_DES_EN").indices.items())]

def score_matrix(w, qs, cs):
    """Scores of the queries qs (rows) against the candidates cs (columns): weights rescaled pair by pair."""
    num = np.zeros((len(qs), len(cs)), np.float32)
    den = np.zeros_like(num)
    for b, wb in w.items():
        if wb > 0:
            both = np.outer(has[b][qs], has[b][cs]) * np.float32(wb)
            num += both * (blocks[b][qs] @ blocks[b][cs].T)
            den += both
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def not_allowed(qs, cs):
    """The query itself and the other colours of its model."""
    out = qs[:, None] == cs[None, :]
    for v in MODEL_CODES:
        out |= (v[qs][:, None] == v[cs][None, :]) & (v[qs][:, None] != "")
    return out

def top_k(w, queries=None, k=10):
    """The k best candidates of every query (or only of `queries`, row numbers): one row per (query, rank)."""
    wanted = np.ones(len(items), bool) if queries is None else np.isin(np.arange(len(items)), queries)
    out = []
    for pool in CATEGORY_POOLS:
        qs_all = pool[wanted[pool]]
        for start in range(0, len(qs_all), 1024):              # 1,024 queries at a time keeps the memory small
            qs = qs_all[start:start + 1024]
            s = np.round(score_matrix(w, qs, pool).astype(np.float64), 6)
            s[not_allowed(qs, pool)] = -np.inf
            order = np.argsort(-s, axis=1, kind="stable")[:, :k]  # the pool is in key order: ties go to the lower key
            best = np.take_along_axis(s, order, axis=1)
            r, c = np.nonzero(np.isfinite(best))
            out.append(pd.DataFrame({"q": qs[r], "rank": c + 1, "c": pool[order[r, c]], "score": best[r, c]}))
    return pd.concat(out, ignore_index=True)

def contributions(w, i, j):
    """Each block's share of the score of the pair (i, j); the shares add up to the score."""
    used = [b for b in BLOCKS if w[b] > 0 and has[b][i] and has[b][j]]
    total = sum(w[b] for b in used)
    return {b: w[b] * float(blocks[b][i] @ blocks[b][j]) / total for b in used}

r = items.iloc[q]
print(f"query {r[KEY]}: {r['GFS_DES_EN']}, {r['CLR_DES']}, {r['PRICE_BASE_W_VAT']} € | weights set by judgement")
first = top_k(W_JUDGEMENT, [q], k=4)
pd.DataFrame([{"rank": t.rank, "candidate": items[KEY].iat[t.c], "score": t.score,
               **{b: round(v, 3) for b, v in contributions(W_JUDGEMENT, q, t.c).items()}} for t in first.itertuples()]).set_index("rank")

## 12. How to judge a model without a right answer

Nobody has labelled which products are "similar", so the model is judged by **three families of signals**, each
measured on the top 4 (the brief asks for at least 4):

| Signal | Family | What it measures |
|---|---|---|
| `hit@4` | catalogue | the query has a **known similar** product (same family › sub-family, `THEME` and `CLR_DES`, another model) and it is in the top 4 |
| `same_subfamily`, `same_colour`, `same_finishing`, `same_material` | catalogue | share of the top 4 that agree with the query (pairs where both have the value; the original values, not the filled ones) |
| `visual@4` | visual | share of the top 4 among the query's 20 nearest **photos** (same category, other models) |
| `sales_corr` | commercial | rank correlation (Spearman) between the query's sales percentile in its category and the mean of its top 4. Sales are **not** used by the model, so this is an independent check: similar products sell alike |
| `price_ratio` | – | median of the higher / lower price in the top 4 (1.00 = same price) |

The **objective** is the mean of the three families (catalogue = mean of its five signals). Each family favours one
block: catalogue the categorical block, visual the image block. So no single signal decides; section 13 uses this.

**No overfitting.** The models (`PROD_REF`) are split in two halves at random (fixed seed): the weights are chosen on
the **tuning half**, and every number reported is measured on the **test half**. A **random** choice of 4 products of
the same category is the floor.


In [ ]:
refs = np.sort(items["PROD_REF"].astype(str).unique())
tune_refs = set(np.random.default_rng(0).permutation(refs)[: len(refs) // 2])
tune = items["PROD_REF"].astype(str).isin(tune_refs).to_numpy()
TUNE_Q, TEST_Q = np.flatnonzero(tune), np.flatnonzero(~tune)

# "known similar": same category › family › sub-family, theme and colour, shared by at least two models
GROUP_COLS = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "THEME", "CLR_DES"]
full = items[GROUP_COLS].notna().all(axis=1)
gid = pd.Series(-1, index=items.index)
gid[full] = items.loc[full].groupby(GROUP_COLS, sort=True).ngroup()
models_in_group = items.loc[full].groupby(gid[full])["PROD_REF"].transform("nunique")
gid[full & models_in_group.reindex(items.index).lt(2)] = -1
gid = gid.to_numpy()

# the 20 nearest photos of every product with a trusted photo
photo = has["image"] > 0
visual_pool = {}
for pool in CATEGORY_POOLS:
    pool = pool[photo[pool]]
    for start in range(0, len(pool), 1024):
        qs = pool[start:start + 1024]
        s = np.round((image_m[qs] @ image_m[pool].T).astype(np.float64), 6)
        s[not_allowed(qs, pool)] = -np.inf
        for qi, row in zip(qs, np.argsort(-s, axis=1, kind="stable")[:, :20]):
            visual_pool[qi] = set(pool[row])

material_csv = items["MATERIAL"].where(~items[KEY].isin(filled.loc[filled["column"] == "MATERIAL", KEY]))
AGREE = {"same_family": items["GFA_DES_EN"],
         "same_subfamily": items["GFA_DES_EN"].astype("string") + " › " + items["GFS_DES_EN"].astype("string"),
         "same_colour": items["CLR_DES"], "same_finishing": items["FINISHING"], "same_material": material_csv}
AGREE = {n: v.astype("string").to_numpy(object, na_value=None) for n, v in AGREE.items()}
sales = items["sales_pct_in_cat"].to_numpy(float)
price = items["PRICE_BASE_W_VAT"].to_numpy(float)

def evaluate(nb):
    """Every signal for one set of neighbours (columns q, rank, c)."""
    top = nb[nb["rank"] <= 4]
    qi, ci = top["q"].to_numpy(), top["c"].to_numpy()
    m = {}
    known = gid[qi] >= 0
    m["hit@4"] = pd.Series(known & (gid[qi] == gid[ci])).groupby(qi).any()[pd.Series(known).groupby(qi).first()].mean()
    for name, v in AGREE.items():
        both = pd.notna(v[qi]) & pd.notna(v[ci])
        m[name] = (v[qi][both] == v[ci][both]).mean()
    m["visual@4"] = np.mean([c in visual_pool[a] for a, c in zip(qi, ci) if a in visual_pool and c in visual_pool])
    per_q = pd.DataFrame({"own": sales[qi], "nb": sales[ci]}).groupby(qi).agg(own=("own", "first"), nb=("nb", "mean")).dropna()
    m["sales_corr"] = per_q["own"].rank().corr(per_q["nb"].rank())
    m["price_ratio"] = np.nanmedian(np.maximum(price[qi], price[ci]) / np.minimum(price[qi], price[ci]))
    m["catalogue"] = np.mean([m[s] for s in ["hit@4", "same_subfamily", "same_colour", "same_finishing", "same_material"]])
    m["visual"], m["commercial"] = m["visual@4"], m["sales_corr"]
    m["objective"] = np.mean([m["catalogue"], m["visual"], m["commercial"]])
    return m

def random_top4(queries, seed=0):
    rng, rows = np.random.default_rng(seed), []
    for pool in CATEGORY_POOLS:
        for qi in queries[np.isin(queries, pool)]:
            allowed = pool[~not_allowed(np.array([qi]), pool)[0]]
            rows += [(qi, r + 1, ci) for r, ci in enumerate(rng.choice(allowed, size=4, replace=False))]
    return pd.DataFrame(rows, columns=["q", "rank", "c"])

SHOW = ["hit@4", "same_family", "same_subfamily", "same_colour", "same_finishing", "same_material", "visual@4",
        "sales_corr", "price_ratio", "catalogue", "objective"]
PCT = {c: "{:.0%}" for c in SHOW if c not in ("sales_corr", "price_ratio", "objective")} | {
    "sales_corr": "{:.3f}", "price_ratio": "{:.2f}", "objective": "{:.3f}"}
results = pd.DataFrame({"random (same category)": evaluate(random_top4(TEST_Q)),
                        "weights set by judgement": evaluate(top_k(W_JUDGEMENT, TEST_Q, k=4))}).T
print(f"tuning half: {len(TUNE_Q):,} queries | test half: {len(TEST_Q):,} queries, "
      f"{(gid[TEST_Q] >= 0).sum():,} of them with a known similar product")
results[SHOW].style.format(PCT)

## 13. Choosing the weights, step 1: tuned on the signals

A blind search on the objective moves almost all the weight to the photo: `visual@4` rises and the material and
colour agreement fall. That is a **trade-off between families**, and the signals cannot decide it (each one favours
its own block). So the search below only accepts a change that is better without being worse:

- **coordinate search** on the tuning half: each block weight in turn is multiplied by 0, 0.5, 1.5 and 2;
- a change is kept only if the objective rises by at least 0.005 **and** no single signal (`hit@4`, the four
  agreements, `visual@4`, `sales_corr`) drops by more than 0.01;
- at most two passes over the six blocks.


In [ ]:
GUARDED = ["hit@4", "same_subfamily", "same_colour", "same_finishing", "same_material", "visual@4", "sales_corr"]

def normalised(w):
    total = sum(w.values())
    return {b: round(v / total, 4) for b, v in w.items()}

def better(new, old):
    return new["objective"] >= old["objective"] + 0.005 and all(new[s] >= old[s] - 0.01 for s in GUARDED)

best = dict(W_JUDGEMENT)
best_m = evaluate(top_k(best, TUNE_Q, k=4))
log = [{"change": "start", **normalised(best), "objective": best_m["objective"]}]
for _ in range(2):
    improved = False
    for b in sorted(best):
        for f in (0.0, 0.5, 1.5, 2.0):
            trial = {**best, b: best[b] * f if best[b] > 0 else (0.1 if f > 1 else 0.0)}
            if trial == best:
                continue
            m = evaluate(top_k(trial, TUNE_Q, k=4))
            if better(m, best_m):
                best, best_m, improved = trial, m, True
                log.append({"change": f"{b} ×{f:g}", **normalised(best), "objective": m["objective"]})
    if not improved:
        break
W_TUNED = normalised(best)
pd.DataFrame(log).style.format({"objective": "{:.3f}", **{b: "{:.3f}" for b in BLOCKS}})

The search raises the photo and the categorical attributes together (0.60 → 0.72 of the weight) and halves the
palette, which mostly repeats what the photo shows. How much the **photo** should weigh against the attributes is
still open. Moving only the image weight (the other blocks keep their proportions) shows the trade-off on the test
half:


In [ ]:
def with_image(w, w_image):
    rest = {b: v for b, v in w.items() if b != "image"}
    return {"image": w_image, **{b: (1 - w_image) * v / sum(rest.values()) for b, v in rest.items()}}

sweep = pd.DataFrame({w_img: evaluate(top_k(with_image(W_TUNED, w_img), TEST_Q, k=4)) for w_img in (0.0, 0.2, 0.4, 0.6, 0.8)}).T
sweep.index.name = "image weight"
fig, ax = plt.subplots(figsize=(7, 3.2))
for col, colour in [("visual@4", "#1f5f8b"), ("same_material", "#e08a00"), ("same_colour", "#6a9f3a"), ("sales_corr", "#888")]:
    ax.plot(sweep.index, sweep[col], "o-", color=colour, label=col)
ax.set_xlabel("weight of the image block")
ax.set_title("More weight on the photo: more visual agreement, less material agreement")
ax.legend(fontsize=8, ncol=2)
plt.show()

## 14. Choosing the weights, step 2: learned from the team's judgement

The trade-off was decided by **people**. Each of the five members sorted products in a small browser app (Sprint 2
work, done outside this part; the answers are saved and loaded here):

1. **140 queries**, chosen at random per category (16 with a trusted photo and 4 without).
2. For each query a **fixed pool** of about 9 candidates: the top 5 at image weights 0, 0.3 and 0.6, plus 2 random
   products of the category. The pool never depends on the weights being learned, so there is no feedback loop.
3. The candidates are shown **shuffled and blind** (no score, no source). The member drags them into *Similar*
   (best first) or *Not similar*.
4. **10 queries** are sorted by every member, to measure how much people agree; **20%** of the queries are kept
   apart (held out) and never used for learning.

`outputs/phase4/feedback_pool.json` has the pools, with the six block similarities of every candidate;
`outputs/phase4/feedback_<n>.jsonl` has each member's answers (the latest answer to a query counts).


In [ ]:
fb_pool = json.loads((ROOT / "outputs" / "phase4" / "feedback_pool.json").read_text(encoding="utf-8"))
latest = {}
for path in sorted((ROOT / "outputs" / "phase4").glob("feedback_*.jsonl")):
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            a = json.loads(line)
            latest[(a["member"], a["query"])] = a
answers = [latest[k] for k in sorted(latest)]

pair_row, sims = {}, []
for pq in fb_pool["queries"]:
    for cand in pq["candidates"]:
        pair_row[(pq["query"], cand["key"])] = len(sims)
        sims.append([np.nan if cand["sim"][b] is None else cand["sim"][b] for b in BLOCKS])
S = np.asarray(sims, float)
H = np.isfinite(S).astype(float)          # 1 where both products have the block
S = np.nan_to_num(S)
held_out = {pq["query"] for pq in fb_pool["queries"] if pq["holdout"]}

def preferences(ans):
    """Every "similar" product beats the similar ones below it and every "not similar" one: (winner, loser) rows."""
    win, lose = [], []
    for a in ans:
        sim = [pair_row[(a["query"], c)] for c in a["similar"]]
        nots = [pair_row[(a["query"], c)] for c in a["not"]]
        for i, x in enumerate(sim):
            for y in sim[i + 1:] + nots:
                win.append(x)
                lose.append(y)
    return np.asarray(win, int), np.asarray(lose, int)

train = [a for a in answers if a["query"] not in held_out]
test = [a for a in answers if a["query"] in held_out]
win, lose = preferences(train)
print(f"{len(answers)} sorted queries by members {sorted({a['member'] for a in answers})} | "
      f"learning: {len({a['query'] for a in train})} queries, {len(win):,} preferences | held out: {len(test)} answers")

**Bradley–Terry model.** The probability that a member prefers candidate *a* to *b* is modelled as
σ(β · (score(a) − score(b))), where σ is the logistic function and *score* is exactly the model of section 11, with
weights w = softmax(θ) (positive, summing to 1). The weights that make the members' preferences most likely are found
by gradient descent (Adam, 1,500 fixed steps: deterministic). A **prior** pulls θ towards the weights of step 1
(normal, sd 0.5 on the log-weights), so a weight moves by a factor of about 1.6 only with clear evidence.


In [ ]:
W0 = np.array([W_TUNED[b] for b in BLOCKS])
PRIOR_SD, BETA0 = 0.5, 20.0

def model_scores(w, s=S, h=H):
    den = h @ w
    return np.divide((h * s) @ w, den, out=np.zeros(len(s)), where=den > 0)

def softmax(t):
    e = np.exp(t - t.max())
    return e / e.sum()

def loss_and_grad(p, theta0):
    theta, gamma = p[:-1], p[-1]
    w, beta = softmax(theta), np.exp(gamma)
    den = H @ w
    sc = np.divide((H * S) @ w, den, out=np.zeros(len(S)), where=den > 0)
    d = sc[win] - sc[lose]
    loss = np.logaddexp(0, -beta * d).sum() + ((theta - theta0) ** 2).sum() / (2 * PRIOR_SD ** 2) + (gamma - np.log(BETA0)) ** 2 / 2
    g_d = -beta / (1 + np.exp(np.clip(beta * d, -50, 50)))
    ds_dw = np.divide(H * S - sc[:, None] * H, den[:, None], out=np.zeros_like(S), where=den[:, None] > 0)
    g_w = (g_d[:, None] * (ds_dw[win] - ds_dw[lose])).sum(0)
    g_theta = w * (g_w - w @ g_w) + (theta - theta0) / PRIOR_SD ** 2
    g_gamma = (g_d * d).sum() + (gamma - np.log(BETA0))
    return loss, np.append(g_theta, g_gamma)

theta0 = np.log(W0 / W0.sum())
p, m1, m2 = np.append(theta0, np.log(BETA0)), np.zeros(len(BLOCKS) + 1), np.zeros(len(BLOCKS) + 1)
for t in range(1, 1501):
    _, g = loss_and_grad(p, theta0)
    m1, m2 = 0.9 * m1 + 0.1 * g, 0.999 * m2 + 0.001 * g ** 2
    p -= 0.05 * (m1 / (1 - 0.9 ** t)) / (np.sqrt(m2 / (1 - 0.999 ** t)) + 1e-8)
W_LEARNED = softmax(p[:-1])

def held_out_accuracy(w):
    """Share of the held-out preferences that the weights order the right way."""
    hw, hl = preferences(test)
    sc = model_scores(w)
    return (sc[hw] > sc[hl]).mean(), len(hw)

acc0, n_pairs = held_out_accuracy(W0)
acc1, _ = held_out_accuracy(W_LEARNED)
print(f"held-out preferences ({n_pairs}) ordered right: step 1 weights {acc0:.1%} -> learned {acc1:.1%}")
pd.DataFrame({"step 1 (signals)": W0, "learned (team)": W_LEARNED}, index=BLOCKS).T.round(3)

The team gives the **photo** much more weight (0.36 → 0.62) and the composition and numeric blocks less. How far can
that be trusted? On the 10 queries everybody sorted, two members order a pair of candidates the same way only about
70% of the time: the learned weights cannot agree with "the team" more than its members agree with each other.


In [ ]:
same = total = 0
by_query = {}
for a in answers:
    by_query.setdefault(a["query"], []).append({c: i for i, c in enumerate(a["similar"])} | {c: len(a["similar"]) for c in a["not"]})
for ranks in by_query.values():
    for i in range(len(ranks)):
        for j in range(i + 1, len(ranks)):
            cands = sorted(ranks[i])
            for x in range(len(cands)):
                for y in range(x + 1, len(cands)):
                    s1 = np.sign(ranks[i][cands[x]] - ranks[i][cands[y]])
                    s2 = np.sign(ranks[j][cands[x]] - ranks[j][cands[y]])
                    if s1 and s2:
                        same, total = same + (s1 == s2), total + 1
print(f"pairs ordered by two members: {total:,}, in the same order: {same / total:.0%}")

**The final weights: a blend.** The fully learned weights order the most held-out preferences right, but they lose
material agreement and pull the prices of the matches apart. So the model uses a point **part of the way** from the
step 1 weights to the learned ones. Each blend is checked on both sides: the team's held-out preferences and the
signals on the test half.


In [ ]:
blend_rows = {}
for share in (0.0, 0.25, 0.5, 0.75, 1.0):
    w = dict(zip(BLOCKS, (1 - share) * W0 + share * W_LEARNED))
    m = evaluate(top_k(w, TEST_Q, k=4))
    blend_rows[f"{share:.0%}"] = {"image weight": w["image"], "held-out preferences right": held_out_accuracy(np.array(list(w.values())))[0],
                                  "same_material": m["same_material"], "visual@4": m["visual@4"],
                                  "sales_corr": m["sales_corr"], "price_ratio": m["price_ratio"]}
blends = pd.DataFrame(blend_rows).T
blends.index.name = "way to the learned weights"
blends.style.format({"image weight": "{:.2f}", "held-out preferences right": "{:.1%}", "same_material": "{:.1%}",
                     "visual@4": "{:.1%}", "sales_corr": "{:.3f}", "price_ratio": "{:.2f}"})

**25%** keeps most of the team's gain at almost no cost to the other signals (the differences between 73% and 75%
are small for about 700 held-out preferences; the material drop beyond 25% is not). These are the weights of the
final model:


In [ ]:
W_FINAL = normalised(dict(zip(BLOCKS, 0.75 * W0 + 0.25 * W_LEARNED)))
pd.DataFrame({"set by judgement": W_JUDGEMENT, "step 1 (signals)": W_TUNED, "step 2 (team, 25%)": W_FINAL}).T.round(3)

## 15. The final model

The top 10 of **every** colourway with the final weights (about 10 seconds), and its evaluation on the test half.


In [ ]:
final = top_k(W_FINAL, k=10)
found = final.groupby("q").size().reindex(range(len(items)), fill_value=0)
coverage = pd.DataFrame({"products": items.groupby("CAT_DES_EN").size(),
                         "with a trusted photo": items.groupby("CAT_DES_EN")["img_trusted"].sum(),
                         "fewer than 4 matches": (found < 4).groupby(items["CAT_DES_EN"]).sum()})
print(f"{final['q'].nunique():,} queries x up to 10 matches = {len(final):,} rows")
coverage

In [ ]:
results.loc["step 1 (signals)"] = evaluate(top_k(W_TUNED, TEST_Q, k=4))
results.loc["final (team, 25%)"] = evaluate(final[np.isin(final["q"], TEST_Q)])
results[SHOW].style.format(PCT)

Against random, the final model finds a known similar product for 92% of the queries (random: 1%), agrees with the
query on the sub-family 90% of the time and on the material 74%, and its matches **sell alike** (correlation 0.62),
although sales are not part of the model. Each weighting step improved the objective on the test half, which the
weights never saw.

**Ablations**: what each block adds. The same final weights with one block left out, and each block alone:


In [ ]:
abl = {"final": W_FINAL, "tabular only (no photo)": {b: (0.0 if b in ("image", "palette") else w) for b, w in W_FINAL.items()}}
abl |= {f"without {b}": {x: (0.0 if x == b else w) for x, w in W_FINAL.items()} for b in BLOCKS}
abl |= {f"only {b}": {x: float(x == b) for x in BLOCKS} for b in BLOCKS}
ablations = pd.DataFrame({name: evaluate(top_k(w, TEST_Q, k=4)) for name, w in abl.items()}).T
ablations[SHOW].style.format(PCT)

- **Each block alone** is weaker than the combination. The categorical block alone scores higher on the catalogue
  signals, because those signals are built from its attributes, but its matches look less alike (`visual@4` 25%
  against 41%) and sell less alike (0.53 against 0.62). The image alone matches photos (100% by definition) but agrees
  on the material only 28% of the time.
- **Without the composition** block the material agreement falls from 74% to 55%; **without the numeric** block the
  sales correlation falls to 0.55 and the prices of the matches drift apart. Every block earns its place.
- **Without a photo** (tabular only) the catalogue agreement stays, and `visual@4` falls to 26%: this is the quality
  the products without a trusted photo get.

## 16. Example: the product of the brief, and why each match was chosen


In [ ]:
top4 = final[(final["q"] == q) & (final["rank"] <= 4)]
show([items[KEY].iat[q]] + items[KEY].iloc[top4["c"]].tolist(), "Final model: the query and its top 4", top4["score"].tolist())
parts = pd.DataFrame([contributions(W_FINAL, q, j) for j in top4["c"]], index=items[KEY].iloc[top4["c"]]).reindex(columns=BLOCKS).fillna(0)
fig, ax = plt.subplots(figsize=(7, 2.6))
left = np.zeros(len(parts))
for b, colour in zip(BLOCKS, ["#1f5f8b", "#e08a00", "#6a9f3a", "#b5651d", "#c94f7c", "#888"]):
    ax.barh(parts.index, parts[b], left=left, color=colour, label=b)
    left += parts[b].to_numpy()
ax.invert_yaxis()
ax.set_xlabel("score = sum of the block contributions")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Why each match: the contribution of each block")
plt.show()
for j in top4["c"]:
    print(f"{items[KEY].iat[j]}: categorical attributes shared =", shared_attributes(cat_m, cat_attrs, q, j))

## Summary

| Block | What it captures | Built |
|---|---|---|
| categorical | family, sub-family, colour name, finishing, theme, … (16 weighted attributes) | here |
| composition | materials, rare ones counting more | here |
| numeric | price within the category, launch date, season | here |
| text | the description's meaning ("Golden Basics" ≈ "Golden Delicates") | sentence model, once |
| image | shape and style of the product crop | CLIP, once |
| palette | the dominant colours of the photo | here (k-means in LAB) |

Also: missing material (apparel) and metal (jewellery) filled only where 5-fold cross-validation shows ≥ 90% precision,
and a reference colour per colour name, measured on the photos, for the explanations.

**Modelling.** score = Σ w_b · sim_b / Σ w_b over the blocks both products have, candidates of the same category and
other models only. The weights were chosen in three steps, each better on the test half:

| Weights | image | categorical | text | composition | palette | numeric | how |
|---|---|---|---|---|---|---|---|
| set by judgement | 0.30 | 0.30 | 0.10 | 0.10 | 0.10 | 0.10 | photo and attributes first |
| step 1 (signals) | 0.36 | 0.36 | 0.08 | 0.08 | 0.04 | 0.08 | guarded coordinate search |
| **final** | **0.42** | **0.33** | **0.08** | **0.07** | **0.03** | **0.07** | Bradley–Terry on 138 sorted queries, 25% of the way |

Every colourway gets at least 4 matches (10 are kept). On the test half the final model finds a known similar product
for 92% of the queries (random 1%), matches the sub-family 90% and the material 74% of the time, and its matches sell
alike (0.62). Each match comes with its block contributions and shared attributes: the base of the explanations.


# Part 3: Sprint 3, explanations, estimates and improvements

**Parfois product similarity, MADSAD case study (FEP, University of Porto).**

The Sprint 2 similarity engine (six blocks, a weighted mean of their similarities, weights learned from the team's
sorting) is the **baseline**: it is frozen and every change is measured against it.

| Section | What |
|---|---|
| 1 | The baseline rebuilt, and the Sprint 3 engine (same weights, FashionSigLIP photo vectors) |
| 2 | How a change is judged |
| 3 | **Explanations** for every similar product: an exact decomposition of the score, and checks |
| 4 | **Sales and price estimates** for a new product, with SHAP explanations |
| 4.3–4.5 | Better models, a better metric, problems in the data, the delivered sales estimate, the largest price errors |
| 5 | Improvements tried against the baseline (kept and rejected) |
| 6 | Photos of a person wearing the item |
| 7 | The Sprint 3 engine against the baseline |

Every step is written here with pandas, NumPy, matplotlib, Pillow and LightGBM. Steps that need large neural networks
or long runs were run once and their saved results are loaded.


## 1. The baseline (Sprint 2) and the Sprint 3 engine

The blocks are rebuilt exactly as in Sprint 2, so this part stands on its own. The colour palettes are loaded
instead of being measured again on every photo.


In [ ]:
CODES = {c: str for c in ["PROD_REF", "PROD_REF_EQUIV", "SEA_COD", "CAT_COD", "GFA_COD", "GFS_COD"]}
items = pd.read_csv(ROOT / "outputs" / "phase1" / "items_clean.csv", encoding="utf-8-sig", dtype=CODES, low_memory=False)
items["STORE_DATE_FINAL"] = pd.to_datetime(items["STORE_DATE_FINAL"])
for c in ["img_trusted", "img_colour_trusted"]:
    items[c] = items[c].astype(str).eq("True")
items = items.sort_values(KEY, kind="stable").reset_index(drop=True)
print(f"{len(items):,} colourways x {items.shape[1]} columns | photos trusted: {items['img_trusted'].sum():,}")
items[[KEY, "CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CLR_DES", "FINISHING", "COMPOSITION", "PRICE_BASE_W_VAT"]].head()

In [ ]:
def normalise_rows(m):
    norms = np.linalg.norm(m, axis=1, keepdims=True)
    return np.divide(m, norms, out=np.zeros_like(m), where=norms > 0).astype(np.float32)

def shared_attributes(m, attrs, i, j):
    """How much each attribute adds to the similarity of rows i and j (the parts add up to the score)."""
    out = {}
    for name, (start, stop) in attrs.items():
        part = float(m[i, start:stop] @ m[j, start:stop])
        if part > 1e-6:
            out[name] = round(part, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def neighbours(m, i, k=4, same_category=True):
    """Top k rows by cosine similarity: same category, other models only (another colour of the same model is
    trivially similar)."""
    s = m @ m[i]
    if same_category:
        s[(items["CAT_DES_EN"] != items["CAT_DES_EN"].iat[i]).to_numpy()] = -1
    s[(items["PROD_REF"] == items["PROD_REF"].iat[i]).to_numpy()] = -1
    top = np.argsort(-s, kind="stable")[:k]
    return top, s[top]

In [ ]:
FINISH_FAMILY = {
    "gold": ["Golden", "Worn Gold", "Brushed Golden", "Antik Golden", "Gold Plated", "Matte Golden", "Burnish Golden",
             "Light Golden", "Worn Yellow Gold", "Matt Brushed Gold", "Scratched Gold"],
    "rose gold": ["Rose Gold", "Brushed Rose Gold", "Worn Rose God"],
    "silver": ["Rodhium", "Silver", "Worn Rhodium", "Worn Silver", "Brushed Silver", "Antik Silver", "Brushed Rhodium",
               "Matte Silver", "Silver Plated"],
    "black": ["Black Nikel", "Black Plating"],
    "other": ["Other", "Multifinishing", "Cooper"],
}
FAMILY_OF = {v: fam for fam, vals in FINISH_FAMILY.items() for v in vals}
items["FINISH_FAMILY"] = items["FINISHING"].map(FAMILY_OF)

# attribute -> (source columns forming a path, weight)
CATEGORICAL = {
    "CAT": (["CAT_DES_EN"], 1.0),
    "GFA": (["CAT_DES_EN", "GFA_DES_EN"], 3.0),
    "GFS": (["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN"], 3.0),
    "CATEGORY_MATRIX": (["CATEGORY_MATRIX"], 1.5),
    "CLR_DES": (["CLR_DES"], 2.0),
    "CLR_TYPE": (["CLR_TYPE"], 0.5),
    "FINISHING": (["FINISHING"], 1.5),
    "FINISH_FAMILY": (["FINISH_FAMILY"], 1.0),
    "MATERIAL": (["MATERIAL"], 1.0),
    "PRINT_TYPE": (["PRINT_TYPE"], 1.0),
    "DIMENSION": (["DIMENSION"], 1.0),
    "OUTFIT": (["OUTFIT"], 1.0),
    "THEME": (["THEME"], 1.5),
    "FASHIONTYPE": (["FASHIONTYPE"], 0.5),
    "PROD_SEG": (["PROD_SEG"], 0.5),
    "NUMBER_OF_UNITS": (["NUMBER_OF_UNITS"], 0.5),
}

def categorical_block(df, spec=CATEGORICAL):
    parts, attrs, start = [], {}, 0
    for name, (cols, weight) in spec.items():
        values = df[cols[0]].astype("string")
        for c in cols[1:]:
            values = values + " › " + df[c].astype("string")       # missing if any part is missing
        levels = sorted(values.dropna().unique())
        codes = pd.Categorical(values, categories=levels).codes
        onehot = np.zeros((len(df), len(levels)), np.float32)
        onehot[np.flatnonzero(codes >= 0), codes[codes >= 0]] = np.sqrt(weight)
        parts.append(onehot)
        attrs[name] = (start, start + len(levels))
        start += len(levels)
    return normalise_rows(np.hstack(parts)), attrs

In [ ]:
def composition_block(df):
    tokens = df["COMPOSITION"].fillna("").map(lambda s: sorted({t.strip() for t in str(s).split(";") if t.strip()}))
    vocab = sorted({t for ts in tokens for t in ts})
    col = {t: j for j, t in enumerate(vocab)}
    counts = pd.Series([t for ts in tokens for t in ts]).value_counts()
    idf = {t: float(np.log((1 + len(df)) / (1 + counts[t])) + 1) for t in vocab}
    m = np.zeros((len(df), len(vocab)), np.float32)
    for i, ts in enumerate(tokens):
        for t in ts:
            m[i, col[t]] = idf[t]
    return normalise_rows(m), {t: (j, j + 1) for t, j in col.items()}, counts, idf

comp, comp_attrs, counts, idf = composition_block(items)

In [ ]:
PRICE_SIGMA, LAUNCH_SIGMA = 0.10, 1.5          # in percentile points / in months
NUMERIC = {"price": 2.0, "launch": 1.0, "season": 0.5}

def soft_bands(x, centres, sigma):
    m = np.exp(-((x[:, None] - centres[None, :]) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    m[np.isnan(x)] = 0
    return normalise_rows(m)

def numeric_block(df):
    price_pct = df.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].rank(pct=True, method="average").to_numpy(float)
    months = ((df["STORE_DATE_FINAL"] - df["STORE_DATE_FINAL"].min()).dt.days / 30.44).to_numpy(float)
    # centres run 3σ past both ends, so values at the edges are compared like the others
    price_c = np.arange(-3 * PRICE_SIGMA, 1 + 3 * PRICE_SIGMA + 1e-9, PRICE_SIGMA / 2)
    launch_c = np.arange(np.floor(np.nanmin(months) - 3 * LAUNCH_SIGMA), np.ceil(np.nanmax(months) + 3 * LAUNCH_SIGMA) + 1e-9,
                         LAUNCH_SIGMA / 2)
    seasons = sorted(df["SEA_COD"].dropna().unique())
    codes = pd.Categorical(df["SEA_COD"], categories=seasons).codes
    season = np.zeros((len(df), len(seasons)), np.float32)
    season[np.flatnonzero(codes >= 0), codes[codes >= 0]] = 1
    parts = {"price": soft_bands(price_pct, price_c, PRICE_SIGMA), "launch": soft_bands(months, launch_c, LAUNCH_SIGMA),
             "season": season}
    attrs, start = {}, 0
    for name, m in parts.items():
        attrs[name] = (start, start + m.shape[1])
        start += m.shape[1]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(NUMERIC[n])) for n, m in parts.items()])), attrs

In [ ]:
def aligned(npy, meta_json):
    keys = json.loads((EMB_DIR / meta_json).read_text(encoding="utf-8"))["keys"]
    emb = np.load(EMB_DIR / npy).astype(np.float32)
    idx = items[KEY].map(pd.Series(range(len(keys)), index=keys))
    out = np.zeros((len(items), emb.shape[1]), np.float32)
    ok = idx.notna().to_numpy()
    out[ok] = emb[idx[ok].astype(int).to_numpy()]
    return out

image_raw = aligned("image_clip.npy", "image_clip.json")
palettes_saved = pd.read_parquet(EMB_DIR / "palettes.parquet").set_index(KEY)
palette_raw = np.stack([np.asarray(v, np.float32) if isinstance(v, (list, np.ndarray)) else np.zeros(24, np.float32)
                        for v in items[KEY].map(palettes_saved["palette_vector"])])
image_raw[~items["img_trusted"].to_numpy()] = 0
palette_raw[~items["img_trusted"].to_numpy()] = 0
yellow = items[KEY].map(pd.Series([sum(s * c[2] for s, c in zip(sh, lab)) for sh, lab in
                                   zip(palettes_saved["shares"], palettes_saved["lab"])], index=palettes_saved.index)).to_numpy()

VOTE_WEIGHTS = {"MATERIAL": {"categorical": 1.0, "composition": 3.0, "image": 1.0, "palette": 0.5},
                "FINISH_FAMILY": {"categorical": 1.0, "composition": 1.0, "image": 2.0, "palette": 2.0}}

def vote_features(target):
    spec = {k: v for k, v in CATEGORICAL.items() if k not in ("FINISHING", "MATERIAL", "FINISH_FAMILY")}
    parts = {"categorical": categorical_block(items, spec)[0], "composition": comp, "image": image_raw, "palette": palette_raw}
    w = VOTE_WEIGHTS[target]
    return normalise_rows(np.hstack([m * np.float32(np.sqrt(w[n])) for n, m in parts.items()]))

def knn_vote(Xq, Xt, yt, k=15):
    pred, conf = np.empty(len(Xq), dtype=object), np.zeros(len(Xq))
    for s in range(0, len(Xq), 512):
        sim = Xq[s:s + 512] @ Xt.T
        top = np.argsort(-sim, axis=1, kind="stable")[:, :k]
        for i, row in enumerate(top):
            votes = pd.Series(np.clip(sim[i, row], 0, None)).groupby(yt[row]).sum()
            if votes.sum() > 0:
                best = votes.sort_index().idxmax()
                pred[s + i], conf[s + i] = best, votes[best] / votes.sum()
    return pred, conf

def photo_agrees(pred, yel):
    pred, yel = np.asarray(pred, dtype=object), np.asarray(yel, float)
    return ~(((pred == "gold") & ~(yel >= 3.0)) | ((pred == "silver") & (yel > 6.0)))

In [ ]:
TARGETS = {"MATERIAL": "Apparel", "FINISH_FAMILY": "Jewellery"}
filled, evidence = [], {}
for col, cat in TARGETS.items():
    X = vote_features(col)
    in_cat = items["CAT_DES_EN"].eq(cat)
    lab = (in_cat & items[col].notna()).to_numpy()
    y, groups = items.loc[lab, col].to_numpy(), items.loc[lab, "PROD_REF"].to_numpy()

    # cross-validation, folds by model
    rng = np.random.default_rng(0)
    models = np.unique(groups)
    fold = pd.Series(rng.integers(0, 5, len(models)), index=models).loc[groups].to_numpy()
    pred, conf = np.empty(len(y), dtype=object), np.zeros(len(y))
    for f in range(5):
        pred[fold == f], conf[fold == f] = knn_vote(X[lab][fold == f], X[lab][fold != f], y[fold != f])
    ok_photo = photo_agrees(pred, yellow[lab]) if col == "FINISH_FAMILY" else np.ones(len(y), bool)
    kept = (conf >= 0.8) & ok_photo
    prec = pd.Series(pred[kept] == y[kept]).groupby(pred[kept]).agg(["mean", "size"])
    allowed = set(prec.index[(prec["mean"] >= 0.9) & (prec["size"] >= 10)])
    evidence[col] = {"products with the value": len(y), "accuracy (all)": (pred == y).mean(),
                     "share kept (vote ≥ 80%)": kept.mean(), "accuracy (kept)": (pred == y)[kept].mean(),
                     "values allowed": ", ".join(sorted(allowed))}

    # fill: trusted photo, value missing
    todo = in_cat & items[col].isna() & items["img_trusted"]
    guess = pd.Series(np.nan, index=items.index[todo], dtype=object)
    method = pd.Series("", index=guess.index)
    if col == "MATERIAL":
        vocab = set(items.loc[in_cat, col].dropna())
        sib = items[items[col].notna()].groupby("PROD_REF")[col].agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
        single = items["COMPOSITION"].fillna("").map(
            lambda s: [t.strip() for t in s.split(";") if t.strip()]).map(lambda t: t[0] if len(t) == 1 and t[0] in vocab else np.nan)
        for name, values in (("sibling", items["PROD_REF"].map(sib)), ("composition", single)):
            free = guess.isna() & values.loc[guess.index].notna()
            guess[free], method[free] = values.loc[guess.index[free]], name
    free = guess.index[guess.isna()]
    p, c = knn_vote(X[free.to_numpy()], X[lab], y)
    keep = (c >= 0.8) & np.isin(p, list(allowed))
    if col == "FINISH_FAMILY":
        keep &= photo_agrees(p, yellow[free.to_numpy()])
    guess[free[keep]], method[free[keep]] = p[keep], "neighbours"
    done = guess.notna()
    filled.append(pd.DataFrame({KEY: items.loc[guess.index[done], KEY], "column": col, "value": guess[done],
                                "method": method[done]}))
filled = pd.concat(filled, ignore_index=True)
print(pd.DataFrame(evidence).T.round(3).to_string())

In [ ]:
print(filled.groupby(["column", "method"]).size().rename("values filled").to_string())
for col in TARGETS:
    sub = filled[filled["column"] == col].set_index(KEY)["value"]
    items[col] = items[col].where(items[col].notna(), items[KEY].map(sub))
print("\nfilled now:", {col: f"{items.loc[items['CAT_DES_EN'] == cat, col].notna().mean():.0%} of {cat}"
                        for col, cat in TARGETS.items()})

In [ ]:
cat_m, cat_attrs = categorical_block(items)
num_m, num_attrs = numeric_block(items)
print(f"categorical {cat_m.shape} | composition {comp.shape} | numeric {num_m.shape}")

In [ ]:
text_m = normalise_rows(aligned("text.npy", "text.json"))
image_m = normalise_rows(image_raw)
palette_m = normalise_rows(palette_raw.copy())
q = items.index[items[KEY] == "167718_BU"][0]            # the example of the brief
blocks = {"categorical": cat_m, "composition": comp, "numeric": num_m, "text": text_m, "image": image_m}
print({b: m.shape for b, m in blocks.items()}, "| palettes", palette_m.shape)

In [ ]:
palette_m[~items["img_colour_trusted"].to_numpy()] = 0
blocks["palette"] = palette_m
BLOCKS = ["image", "categorical", "text", "composition", "palette", "numeric"]
W_JUDGEMENT = {"image": 0.3, "categorical": 0.3, "text": 0.1, "composition": 0.1, "palette": 0.1, "numeric": 0.1}

has = {b: (np.abs(m).sum(1) > 0).astype(np.float32) for b, m in blocks.items()}
MODEL_CODES = [items[c].astype("string").fillna("").to_numpy(object) for c in ["PROD_REF", "PROD_REF_EQUIV"]]
CATEGORY_POOLS = [np.asarray(ix) for _, ix in sorted(items.groupby("CAT_DES_EN").indices.items())]

def score_matrix(w, qs, cs):
    """Scores of the queries qs (rows) against the candidates cs (columns): weights rescaled pair by pair."""
    num = np.zeros((len(qs), len(cs)), np.float32)
    den = np.zeros_like(num)
    for b, wb in w.items():
        if wb > 0:
            both = np.outer(has[b][qs], has[b][cs]) * np.float32(wb)
            num += both * (blocks[b][qs] @ blocks[b][cs].T)
            den += both
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def not_allowed(qs, cs):
    """The query itself and the other colours of its model."""
    out = qs[:, None] == cs[None, :]
    for v in MODEL_CODES:
        out |= (v[qs][:, None] == v[cs][None, :]) & (v[qs][:, None] != "")
    return out

def top_k(w, queries=None, k=10):
    """The k best candidates of every query (or only of `queries`, row numbers): one row per (query, rank)."""
    wanted = np.ones(len(items), bool) if queries is None else np.isin(np.arange(len(items)), queries)
    out = []
    for pool in CATEGORY_POOLS:
        qs_all = pool[wanted[pool]]
        for start in range(0, len(qs_all), 1024):              # 1,024 queries at a time keeps the memory small
            qs = qs_all[start:start + 1024]
            s = np.round(score_matrix(w, qs, pool).astype(np.float64), 6)
            s[not_allowed(qs, pool)] = -np.inf
            order = np.argsort(-s, axis=1, kind="stable")[:, :k]  # the pool is in key order: ties go to the lower key
            best = np.take_along_axis(s, order, axis=1)
            r, c = np.nonzero(np.isfinite(best))
            out.append(pd.DataFrame({"q": qs[r], "rank": c + 1, "c": pool[order[r, c]], "score": best[r, c]}))
    return pd.concat(out, ignore_index=True)

def contributions(w, i, j):
    """Each block's share of the score of the pair (i, j); the shares add up to the score."""
    used = [b for b in BLOCKS if w[b] > 0 and has[b][i] and has[b][j]]
    total = sum(w[b] for b in used)
    return {b: w[b] * float(blocks[b][i] @ blocks[b][j]) / total for b in used}

The baseline weights are the Sprint 2 result (a 25% blend between the tuned and the team-learned weights):


In [ ]:
W_BASE = {"image": 0.4246, "categorical": 0.3318, "text": 0.0767, "composition": 0.0671, "palette": 0.034, "numeric": 0.066}
t0 = time.time()
base_nb = top_k(W_BASE, k=10)
print(f"baseline: top 10 of {base_nb['q'].nunique():,} products in {time.time() - t0:.0f} s")

# the Sprint 3 engine: same weights, the photo encoded with FashionSigLIP (computed once, section 5.3 and 6)
sig_meta = json.loads((EMB_DIR / "image_fashionsiglip.json").read_text(encoding="utf-8"))
sig = np.load(EMB_DIR / "image_fashionsiglip.npy").astype(np.float32)
idx = items[KEY].map(pd.Series(range(len(sig_meta["keys"])), index=sig_meta["keys"]))
sig_m = np.zeros((len(items), sig.shape[1]), np.float32)
ok = idx.notna().to_numpy() & items["img_trusted"].to_numpy()
sig_m[ok] = sig[idx[ok].astype(int).to_numpy()]
sig_m = normalise_rows(sig_m)
clip_m = image_m
blocks["image"] = sig_m
s3_nb = top_k(W_BASE, k=10)
print(f"Sprint 3 engine: {sig_meta['model']}, {sig_m.shape[1]} numbers per photo; "
      f"top-1 match differs from the baseline for {(s3_nb[s3_nb['rank'] == 1].set_index('q')['c'] != base_nb[base_nb['rank'] == 1].set_index('q')['c']).mean():.0%} of the products")

## 2. How a change is judged

There is no ground truth, so three signals are used, strongest first:

1. **The team's judgements.** Each member sorted the candidates of their queries into *similar* (best first) and *not
   similar*: 138 answers on 99 queries, about 4,000 pairwise preferences. A version of the engine is scored by the share
   it orders right, with the weights fixed and with **5-fold cross-validation over queries**. Uncertainty: a **paired
   bootstrap over queries**.
2. **Labels a change never saw.** For an image encoder: do a photo's 4 nearest photos share its sub-family, material
   and finish?
3. **Guards**: the Sprint 2 metrics on the test half of the models may not fall by more than 1 point. And the brief's
   example: where the engine ranks the 4 products Parfois gave for `167718_BU`.

Ceiling: two team members agree on about 70% of the pairs they both sorted.


In [ ]:
refs = np.sort(items["PROD_REF"].astype(str).unique())
tune_refs = set(np.random.default_rng(0).permutation(refs)[: len(refs) // 2])
tune = items["PROD_REF"].astype(str).isin(tune_refs).to_numpy()
TUNE_Q, TEST_Q = np.flatnonzero(tune), np.flatnonzero(~tune)

# "known similar": same category › family › sub-family, theme and colour, shared by at least two models
GROUP_COLS = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "THEME", "CLR_DES"]
full = items[GROUP_COLS].notna().all(axis=1)
gid = pd.Series(-1, index=items.index)
gid[full] = items.loc[full].groupby(GROUP_COLS, sort=True).ngroup()
models_in_group = items.loc[full].groupby(gid[full])["PROD_REF"].transform("nunique")
gid[full & models_in_group.reindex(items.index).lt(2)] = -1
gid = gid.to_numpy()

# the 20 nearest photos of every product with a trusted photo
photo = has["image"] > 0
visual_pool = {}
for pool in CATEGORY_POOLS:
    pool = pool[photo[pool]]
    for start in range(0, len(pool), 1024):
        qs = pool[start:start + 1024]
        s = np.round((image_m[qs] @ image_m[pool].T).astype(np.float64), 6)
        s[not_allowed(qs, pool)] = -np.inf
        for qi, row in zip(qs, np.argsort(-s, axis=1, kind="stable")[:, :20]):
            visual_pool[qi] = set(pool[row])

material_csv = items["MATERIAL"].where(~items[KEY].isin(filled.loc[filled["column"] == "MATERIAL", KEY]))
AGREE = {"same_family": items["GFA_DES_EN"],
         "same_subfamily": items["GFA_DES_EN"].astype("string") + " › " + items["GFS_DES_EN"].astype("string"),
         "same_colour": items["CLR_DES"], "same_finishing": items["FINISHING"], "same_material": material_csv}
AGREE = {n: v.astype("string").to_numpy(object, na_value=None) for n, v in AGREE.items()}
sales = items["sales_pct_in_cat"].to_numpy(float)
price = items["PRICE_BASE_W_VAT"].to_numpy(float)

def evaluate(nb):
    """Every signal for one set of neighbours (columns q, rank, c)."""
    top = nb[nb["rank"] <= 4]
    qi, ci = top["q"].to_numpy(), top["c"].to_numpy()
    m = {}
    known = gid[qi] >= 0
    m["hit@4"] = pd.Series(known & (gid[qi] == gid[ci])).groupby(qi).any()[pd.Series(known).groupby(qi).first()].mean()
    for name, v in AGREE.items():
        both = pd.notna(v[qi]) & pd.notna(v[ci])
        m[name] = (v[qi][both] == v[ci][both]).mean()
    m["visual@4"] = np.mean([c in visual_pool[a] for a, c in zip(qi, ci) if a in visual_pool and c in visual_pool])
    per_q = pd.DataFrame({"own": sales[qi], "nb": sales[ci]}).groupby(qi).agg(own=("own", "first"), nb=("nb", "mean")).dropna()
    m["sales_corr"] = per_q["own"].rank().corr(per_q["nb"].rank())
    m["price_ratio"] = np.nanmedian(np.maximum(price[qi], price[ci]) / np.minimum(price[qi], price[ci]))
    m["catalogue"] = np.mean([m[s] for s in ["hit@4", "same_subfamily", "same_colour", "same_finishing", "same_material"]])
    m["visual"], m["commercial"] = m["visual@4"], m["sales_corr"]
    m["objective"] = np.mean([m["catalogue"], m["visual"], m["commercial"]])
    return m

def random_top4(queries, seed=0):
    rng, rows = np.random.default_rng(seed), []
    for pool in CATEGORY_POOLS:
        for qi in queries[np.isin(queries, pool)]:
            allowed = pool[~not_allowed(np.array([qi]), pool)[0]]
            rows += [(qi, r + 1, ci) for r, ci in enumerate(rng.choice(allowed, size=4, replace=False))]
    return pd.DataFrame(rows, columns=["q", "rank", "c"])

SHOW = ["hit@4", "same_family", "same_subfamily", "same_colour", "same_finishing", "same_material", "visual@4",
        "sales_corr", "price_ratio", "catalogue", "objective"]
PCT = {c: "{:.0%}" for c in SHOW if c not in ("sales_corr", "price_ratio", "objective")} | {
    "sales_corr": "{:.3f}", "price_ratio": "{:.2f}", "objective": "{:.3f}"}
results = pd.DataFrame({"random (same category)": evaluate(random_top4(TEST_Q)),
                        "weights set by judgement": evaluate(top_k(W_JUDGEMENT, TEST_Q, k=4))}).T
print(f"tuning half: {len(TUNE_Q):,} queries | test half: {len(TEST_Q):,} queries, "
      f"{(gid[TEST_Q] >= 0).sum():,} of them with a known similar product")
results.loc["baseline (Sprint 2)"] = evaluate(base_nb[base_nb["q"].isin(TEST_Q)])
results.loc["Sprint 3 engine"] = evaluate(s3_nb[s3_nb["q"].isin(TEST_Q)])
results[SHOW].round(3)

In [ ]:
fb_pool = json.loads((ROOT / "outputs" / "phase4" / "feedback_pool.json").read_text(encoding="utf-8"))
latest = {}
for path in sorted((ROOT / "outputs" / "phase4").glob("feedback_*.jsonl")):
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            a = json.loads(line)
            latest[(a["member"], a["query"])] = a
answers = [latest[k] for k in sorted(latest)]

pair_row, sims = {}, []
for pq in fb_pool["queries"]:
    for cand in pq["candidates"]:
        pair_row[(pq["query"], cand["key"])] = len(sims)
        sims.append([np.nan if cand["sim"][b] is None else cand["sim"][b] for b in BLOCKS])
S = np.asarray(sims, float)
H = np.isfinite(S).astype(float)          # 1 where both products have the block
S = np.nan_to_num(S)
held_out = {pq["query"] for pq in fb_pool["queries"] if pq["holdout"]}

def preferences(ans):
    """Every "similar" product beats the similar ones below it and every "not similar" one: (winner, loser) rows."""
    win, lose = [], []
    for a in ans:
        sim = [pair_row[(a["query"], c)] for c in a["similar"]]
        nots = [pair_row[(a["query"], c)] for c in a["not"]]
        for i, x in enumerate(sim):
            for y in sim[i + 1:] + nots:
                win.append(x)
                lose.append(y)
    return np.asarray(win, int), np.asarray(lose, int)

train = [a for a in answers if a["query"] not in held_out]
test = [a for a in answers if a["query"] in held_out]
win, lose = preferences(train)
print(f"{len(answers)} sorted answers on {len({a['query'] for a in answers})} distinct queries, members {sorted({a['member'] for a in answers})} | "
      f"learning: {len({a['query'] for a in train})} queries, {len(win):,} preferences | held out: {len(test)} answers")

In [ ]:
def fit_weights(S_, H_, win_, lose_, w0, steps=1500):
    """Bradley-Terry fit of the block weights on (winner, loser) rows, pulled towards w0 (as in Sprint 2)."""
    def lg(p, theta0):
        theta, gamma = p[:-1], p[-1]
        w, beta = np.exp(theta - theta.max()) / np.exp(theta - theta.max()).sum(), np.exp(gamma)
        den = H_ @ w
        sc = np.divide((H_ * S_) @ w, den, out=np.zeros(len(S_)), where=den > 0)
        d = sc[win_] - sc[lose_]
        g_d = -beta / (1 + np.exp(np.clip(beta * d, -50, 50)))
        ds_dw = np.divide(H_ * S_ - sc[:, None] * H_, den[:, None], out=np.zeros_like(S_), where=den[:, None] > 0)
        g_w = (g_d[:, None] * (ds_dw[win_] - ds_dw[lose_])).sum(0)
        return np.append(w * (g_w - w @ g_w) + (theta - theta0) / 0.5 ** 2, (g_d * d).sum() + (gamma - np.log(20.0)))
    theta0 = np.log(np.maximum(w0, 1e-3) / np.maximum(w0, 1e-3).sum())
    p, m1, m2 = np.append(theta0, np.log(20.0)), np.zeros(len(w0) + 1), np.zeros(len(w0) + 1)
    for t in range(1, steps + 1):
        g = lg(p, theta0)
        m1, m2 = 0.9 * m1 + 0.1 * g, 0.999 * m2 + 0.001 * g ** 2
        p -= 0.05 * (m1 / (1 - 0.9 ** t)) / (np.sqrt(m2 / (1 - 0.999 ** t)) + 1e-8)
    e = np.exp(p[:-1] - p[:-1].max())
    return e / e.sum()

def per_query_right(w, S_, ans):
    """Right / total preferences per query for fixed weights."""
    sc = model_scores(w, S_, H)
    out = {}
    for a in ans:
        hw, hl = preferences([a])
        o = out.setdefault(a["query"], [0, 0])
        o[0] += int((sc[hw] > sc[hl]).sum())
        o[1] += len(hw)
    return pd.DataFrame(out, index=["right", "pairs"]).T

def model_scores(w, s, h):
    den = h @ w
    return np.divide((h * s) @ w, den, out=np.zeros(len(s)), where=den > 0)

QUERIES = sorted({a["query"] for a in answers})
FOLD = dict(zip(QUERIES, np.random.default_rng(1).integers(0, 5, len(QUERIES))))
W_START = np.array([fb_pool["weights"][b] for b in BLOCKS])
W_BASE_V = np.array([W_BASE[b] for b in BLOCKS])

def team_score(S_):
    """Team preferences ordered right: fixed baseline weights, and 5-fold CV with refitted weights."""
    fixed = per_query_right(W_BASE_V, S_, answers)
    cv = []
    for f in range(5):
        tr = [a for a in answers if FOLD[a["query"]] != f]
        w_ = fit_weights(S_, H, *preferences(tr), W_START)
        cv.append(per_query_right(w_, S_, [a for a in answers if FOLD[a["query"]] == f]))
    return fixed, pd.concat(cv)

def boot_diff(a, b, n=2000, seed=2):
    """95% interval of (accuracy of b) - (accuracy of a), resampling queries."""
    b = b.loc[a.index]
    rng, d = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        d.append(b["right"].values[i].sum() / b["pairs"].values[i].sum() - a["right"].values[i].sum() / a["pairs"].values[i].sum())
    return np.percentile(d, [2.5, 97.5])

acc = lambda t: t["right"].sum() / t["pairs"].sum()
base_fixed, base_cv = team_score(S)
print(f"baseline on the team's preferences: fixed weights {acc(base_fixed):.1%}, 5-fold CV {acc(base_cv):.1%} "
      f"({base_fixed['pairs'].sum():,} preferences, {len(base_fixed)} queries)")

## 3. Explanations for every similar product

The brief asks for at least 4 similar products **and the characteristics that explain the match**. The score is a
weighted mean of block similarities, and each block is a dot product of named slices, so the explanation is an
**exact decomposition**, not an approximation:

- score = Σ w_b · sim_b / Σ w_b → **the share of each block**;
- sim_categorical = Σ over attributes → **which attributes are shared, and how much each adds**;
- sim_composition = Σ over materials → **shared materials** (rare ones weigh more);
- sim_numeric = price + launch + season.

Differences (price, colour, finish, sub-family) are added from the data; colour names that look alike on the photos are
reported as **close colours**. No surrogate model and no sampling: the explanation cannot say what the score does not
contain.


In [ ]:
ATTR_LABEL = {"GFA": "family", "GFS": "sub-family", "CATEGORY_MATRIX": "planning category", "CLR_DES": "colour",
              "CLR_TYPE": "colour type", "FINISHING": "finish", "FINISH_FAMILY": "metal", "MATERIAL": "main fabric",
              "PRINT_TYPE": "print", "DIMENSION": "size type", "OUTFIT": "outfit", "THEME": "theme",
              "FASHIONTYPE": "fashion type", "PROD_SEG": "segment", "NUMBER_OF_UNITS": "units"}
BLOCK_LABEL = {"image": "photo", "categorical": "catalogue attributes", "text": "description", "composition": "materials",
               "palette": "photo colours", "numeric": "price and launch"}
SLICES = {"categorical": cat_attrs, "composition": comp_attrs, "numeric": num_attrs}
VALUE_COL = {a: cols[-1] for a, (cols, _) in CATEGORICAL.items()}

# colour names measured on the photos (Sprint 2, section 9): reference LAB per name, similarity of two names
ref = pd.read_csv(ROOT / "outputs" / "phase2a" / "colour_reference.csv").dropna(subset=["L"])
lab_ref = ref[["L", "a", "b"]].to_numpy(float) * np.array([0.5, 1, 1])
csim = np.exp(-((lab_ref[:, None] - lab_ref[None]) ** 2).sum(-1) / (2 * 8.0 ** 2))
COLOUR_SIM = {(x, y): csim[i, j] for i, x in enumerate(ref["CLR_DES"]) for j, y in enumerate(ref["CLR_DES"]) if i != j}

def parts(block, i, j):
    m = blocks[block]
    return {a: float(m[i, s:e] @ m[j, s:e]) for a, (s, e) in SLICES[block].items() if abs(m[i, s:e] @ m[j, s:e]) > 1e-9}

def explain(w, i, j):
    """Exact explanation of the score of the pair (query i, candidate j)."""
    contrib = contributions(w, i, j)
    score = sum(contrib.values())
    out = {"score": score, "shares": {BLOCK_LABEL[b]: v / score for b, v in contrib.items()}}
    shared = []
    if "categorical" in contrib:
        sim = float(blocks["categorical"][i] @ blocks["categorical"][j])
        for a, v in sorted(parts("categorical", i, j).items(), key=lambda kv: -kv[1]):
            if a != "CAT":
                shared.append((ATTR_LABEL.get(a, a), items[VALUE_COL[a]].iat[i], contrib["categorical"] * v / sim))
    out["shared"] = shared
    if "composition" in contrib:
        sim = float(comp[i] @ comp[j])
        out["materials"] = [(m_, contrib["composition"] * v / sim) for m_, v in sorted(parts("composition", i, j).items(), key=lambda kv: -kv[1])]
    a_, b_ = items.iloc[i], items.iloc[j]
    diffs = []
    if abs(b_["PRICE_BASE_W_VAT"] - a_["PRICE_BASE_W_VAT"]) > 0.05 * a_["PRICE_BASE_W_VAT"]:
        diffs.append(f"price {b_['PRICE_BASE_W_VAT']:.2f} € vs {a_['PRICE_BASE_W_VAT']:.2f} €")
    if a_["CLR_DES"] != b_["CLR_DES"]:
        s_ = COLOUR_SIM.get((a_["CLR_DES"], b_["CLR_DES"]))
        diffs.append(f"close colours {b_['CLR_DES']} ≈ {a_['CLR_DES']} ({s_:.2f})" if s_ is not None and s_ >= 0.5
                     else f"colour {b_['CLR_DES']} vs {a_['CLR_DES']}")
    for col, lab in [("GFS_DES_EN", "sub-family"), ("FINISHING", "finish")]:
        if pd.notna(a_[col]) and pd.notna(b_[col]) and a_[col] != b_[col]:
            diffs.append(f"{lab} {b_[col]} vs {a_[col]}")
    out["differences"] = diffs
    return out

def summary(e):
    lead = ", ".join(f"{k} {v:.0%}" for k, v in sorted(e["shares"].items(), key=lambda kv: -kv[1])[:3])
    facts = [f"same {a}: {v}" for a, v, _ in e["shared"][:3]]
    if e.get("materials"):
        facts.append("shared materials: " + ", ".join(m_ for m_, _ in e["materials"][:3]))
    return f"{lead}. " + "; ".join(facts) + (". Differs: " + "; ".join(e["differences"][:2]) if e["differences"] else "")

for t in s3_nb[(s3_nb["q"] == q) & (s3_nb["rank"] <= 5)].itertuples():
    print(f"{t.rank}. {items[KEY].iat[t.c]}  {t.score:.3f}  {summary(explain(W_BASE, q, t.c))}")

The brief's example: the photo and the catalogue attributes carry most of each score; the matches share family,
sub-family, planning category and materials with the query, and differ in colour.

**Check 1: completeness.** The parts must add up to the score, for every pair.


In [ ]:
rng = np.random.default_rng(0)
sample = s3_nb[s3_nb["rank"] <= 5].sample(2000, random_state=0)
err_blocks = err_attrs = 0.0
for t in sample.itertuples():
    e = explain(W_BASE, t.q, t.c)
    err_blocks = max(err_blocks, abs(e["score"] - t.score))
    sim = float(blocks["categorical"][t.q] @ blocks["categorical"][t.c])
    err_attrs = max(err_attrs, abs(sum(parts("categorical", t.q, t.c).values()) - sim))
print(f"largest gap on 2,000 pairs: blocks vs score {err_blocks:.1e} | attributes vs categorical similarity {err_attrs:.1e}")

**Check 2: faithfulness**, on the best match of 300 random products (ties counted against the match):

- *Deletion*: remove the block named first, last or a random one, and see how far the match falls.
- *Sufficiency*: keep **only** the two blocks named first and see whether the match stays in the top 5, against two
  random blocks or the two named last. A control keeps the two blocks with the largest global weights (photo and
  attributes).


In [ ]:
def rank_of(w, qi, ci, pool):
    s = np.round(score_matrix(w, np.array([qi]), pool)[0].astype(np.float64), 6)
    s[not_allowed(np.array([qi]), pool)[0]] = -np.inf
    return int((s >= s[pool == ci][0]).sum())

rows = []
for qi in rng.choice(len(items), 300, replace=False):
    first = s3_nb[(s3_nb["q"] == qi) & (s3_nb["rank"] == 1)]
    if first.empty:
        continue
    ci = int(first["c"].iat[0])
    ranked = [b for b, _ in sorted(contributions(W_BASE, qi, ci).items(), key=lambda kv: -kv[1])]
    if len(ranked) < 4:
        continue
    pool = next(p for p in CATEGORY_POOLS if qi in p)
    for label, b in [("named first", ranked[0]), ("named last", ranked[-1]), ("random", ranked[rng.integers(len(ranked))])]:
        rows.append({"test": "deletion: block removed", "variant": label, "new rank": rank_of({**W_BASE, b: 0.0}, qi, ci, pool)})
    heavy = sorted(ranked, key=lambda b: -W_BASE[b])[:2]           # control: the two largest global weights
    differs = set(ranked[:2]) != set(heavy)
    for label, keep in [("named first", ranked[:2]), ("two largest weights (control)", heavy), ("named last", ranked[-2:]),
                        ("random", list(rng.choice(ranked, 2, replace=False)))]:
        w = {b: (W_BASE[b] if b in keep else 0.0) for b in W_BASE}
        r_ = rank_of(w, qi, ci, pool)
        rows.append({"test": "sufficiency: only 2 blocks kept", "variant": label, "new rank": r_})
        if differs and label in ("named first", "two largest weights (control)"):
            rows.append({"test": "sufficiency, only where the explanation names other blocks", "variant": label, "new rank": r_})
ft = pd.DataFrame(rows).groupby(["test", "variant"])["new rank"]
pd.DataFrame({"pairs": ft.size(), "mean rank": ft.mean(), "still first": ft.apply(lambda r: (r == 1).mean()),
              "in the top 5": ft.apply(lambda r: (r <= 5).mean())}).round(2)

**Reading.** The blocks named first behave as expected, but the **control gives the same result**: for almost every
match they are the photo and the attributes. These tests show that the heavy blocks drive the matches; what is
specific to each match is inside the blocks (shared attributes and materials, differences), and that part is exact
(check 1).


**Check 3: agreement with the team.** For the candidates the engine proposed in the team's sorting (random
candidates left out), the mean similarity of each block for *similar* and *not similar*, and Cohen's d.


In [ ]:
random_cands = {(pq["query"], c["key"]) for pq in fb_pool["queries"] for c in pq["candidates"] if c.get("random")}
# the pool stores CLIP photo similarities; the Sprint 3 engine's photo similarity is recomputed for the same pairs
pos_ = pd.Series(np.arange(len(items)), index=items[KEY])
pair_q, pair_c = np.zeros(len(S), int), np.zeros(len(S), int)
for (qk, ck), r_ in pair_row.items():
    pair_q[r_], pair_c[r_] = pos_[qk], pos_[ck]
S_eng = S.copy()
k_img = BLOCKS.index("image")
S_eng[:, k_img] = np.where(H[:, k_img] > 0, np.einsum("ij,ij->i", sig_m[pair_q], sig_m[pair_c]), 0)
rows = []
for a in answers:
    for label, keys_ in (("similar", a["similar"]), ("not similar", a["not"])):
        for c in keys_:
            if (a["query"], c) in random_cands:
                continue
            r_ = pair_row[(a["query"], c)]   # a block missing for one of the two products is left out, not counted as 0
            rows.append({"verdict": label, **{BLOCK_LABEL[b]: (S_eng[r_, k] if H[r_, k] else np.nan) for k, b in enumerate(BLOCKS)}})
ag = pd.DataFrame(rows)
g = ag.groupby("verdict")
tab = g.mean().T
n_, var_ = g.size(), g.var().T
pooled = np.sqrt(((n_["similar"] - 1) * var_["similar"] + (n_["not similar"] - 1) * var_["not similar"]) / (n_.sum() - 2))
tab["Cohen's d"] = (tab["similar"] - tab["not similar"]) / pooled
print("judgements used:", {k: int(v) for k, v in n_.items()})
tab.sort_values("Cohen's d", ascending=False).round(3)

The **photo** separates best what the team calls similar, then the description, the attributes and the materials;
the photo colours separate nothing, as in the brief's example, where the four similar earrings have other colours.


## 4. Price and sales estimates for a new product

The business problem is buying the right quantity of a **new** product and sending it to the right stores. The
similar products and all the product's attributes become an estimate of its sales and of its price, each explained.

**First, the data**: are there products without a price, and what is missing in the sales?


In [ ]:
price_ = items["PRICE_BASE_W_VAT"].astype(float)
qty = items["SALES_QTY"]
cost_ = pd.to_numeric(items["PRICE_LCP"], errors="coerce")
print(f"list price: {price_.isna().sum()} missing, {(price_ <= 0).sum()} at 0 € or less, "
      f"{price_.nunique()} price points from {price_.min():.2f} € to {price_.max():.2f} €")
print(f"sales: {qty.isna().sum()} colourways without a sales row, {(qty == 0).sum()} with 0 units "
      f"({((qty == 0) & (items['SALES_AMT_FX_RATE'] > 0)).sum()} of them with a positive amount)")
print(f"landed cost (PRICE_LCP): missing or <= 0 for {(~(cost_ > 0)).sum()} colourways")
items["launch"] = pd.to_datetime(items["STORE_DATE_PLANNED"])     # the date known when ordering
items["log_sales"] = np.log1p(qty.where(qty > 0))
items["log_cost"] = np.log(cost_.where(cost_ > 0))
sea_ = items["SEA_COD"].astype(str)

# sales are accumulated until the extraction, whose date is not given: products launched after it have no sale yet
start_ = items["STORE_DATE_FINAL"]
late_ = start_ >= "2024-09-01"
no_sale = (qty[late_].fillna(0) == 0).groupby(start_[late_].dt.to_period("W").dt.start_time).agg(["size", "mean"])
extraction = no_sale.index[(no_sale["mean"] >= 0.5) & (no_sale["size"] >= 20)][0]
print(f"\nextraction: week of {extraction:%d %b %Y} (the first launch week where most products have no sale)")
month_ = items["STORE_DATE_FINAL"].dt.strftime("%Y-%m")
cut = items.assign(month=month_)[sea_.isin(["241", "242"])].groupby(["SEA_COD", "month"])["SALES_QTY"].agg(["size", "median"])
print("\nmedian units by launch month (seasons 241 and 242):")
print(cut[cut["size"] >= 5].round(0).astype(int).T.to_string())
outlet = pd.to_datetime(items["STORE_DATE_OUTLET_REAL"])
o241 = outlet[sea_.eq("241")]
gone = sea_.eq("241") & (outlet < extraction)
print(f"season 241: {o241.notna().mean():.0%} with an outlet date, {(o241 < extraction).mean():.0%} already in the outlet at "
      f"the extraction (the later dates are planned); those stayed a median of "
      f"{((outlet - start_)[gone].dt.days / 7).median():.0f} weeks at full price")

print("\nmedian markup (list price / landed cost) by category and season:")
print((price_ / cost_.where(cost_ > 0)).groupby([items["CAT_DES_EN"], sea_]).median().unstack().round(2).to_string())

- **Every product has a list price** (no 0 € and no missing value). Missing: the landed cost for a few products and
  the sales of 379 (no row or 0 units). A product without sales is *unknown*, not zero: out of training and evaluation.
- **The sales were extracted in early November 2024** (from that week almost no launch has a sale). Season 242 had not
  finished selling (October launches: about 140 units, against 3,300 for a 241 launch). Season 241 had sold for 3 to 12
  months (28% already in the outlet); its late launches show no cut (bias +3%), so its totals are close to whole lives.
  Season 232 had sold for more than a year.
- **Sales = whole-life units**: **train 232 → the new models of 241**, split by model into a validation half (choices)
  and a test half (reported once). 242 only checks the ranking among products launched in the same month. The weeks on
  sale are used only when learning (sections 4.3–4.4).
- **Price = landed cost × markup.** The cost is known when the price is set and carries the price rise between
  seasons; the markup is stable within a category. Validation 232 → 241, test 232 + 241 → 242.
- **Buyer decisions are left out** (`PUR_BET` purchase depth, `TYPOLOGY` store tiers): they are the buyer's own
  expectation of the sales. They are a benchmark and a second opinion (section 4.1b).
- **Features**, by a rule fixed before the choice: sales use the attributes and the photo components, price the
  attributes. Description, materials and launch month help in neither.


In [ ]:
import lightgbm as lgb

FC_CATEGORICAL = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN", "CATEGORY_MATRIX", "CLR_DES", "CLR_TYPE", "FINISHING",
                  "MATERIAL", "PRINT_TYPE", "DIMENSION", "OUTFIT", "FASHIONTYPE", "PROD_SEG", "NUMBER_OF_UNITS",
                  "COLLECTION_TYPE", "MADE_IN", "SUP_COD"]
LGB = dict(objective="regression_l2", learning_rate=0.03, num_leaves=31, min_child_samples=20, feature_fraction=0.8,
           bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, n_estimators=600, random_state=0, deterministic=True,
           force_col_wise=True, num_threads=4, verbose=-1)

def model_groups():
    """One id per model, joining PROD_REF and PROD_REF_EQUIV (re-coded items are the same model)."""
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(items["PROD_REF"].astype(str), items["PROD_REF_EQUIV"].fillna(items["PROD_REF"]).astype(str)):
        parent[find(a)] = find(b)
    return np.array([find(a) for a in items["PROD_REF"].astype(str)])
GROUPS = model_groups()

def make_split(train_seasons, test_season, half=None):
    """Train: the given seasons. Test: the new models of a later season; `half` keeps one random half of them."""
    tr = sea_.isin(train_seasons).to_numpy()
    seen_ = set(items.loc[tr, "PROD_REF"].astype(str))
    te = sea_.eq(test_season).to_numpy() & ~items["PROD_REF"].astype(str).isin(seen_).to_numpy()
    if half is not None:
        uniq = np.unique(GROUPS)
        side = dict(zip(uniq, np.random.default_rng(0).permutation(len(uniq)) % 2))
        te &= np.array([side[g] == half for g in GROUPS])
    return tr, te

SALES_SPLITS = {"validation (232 -> half of the new 241 models)": make_split(["232"], "241", 0),
                "test (232 -> the other half)": make_split(["232"], "241", 1)}
RANK_242 = make_split(["232", "241"], "242")          # sales cut by the extraction: ranking only
PRICE_SPLITS = {"validation (232 -> 241)": make_split(["232"], "241"),
                "test (232 + 241 -> 242)": make_split(["232", "241"], "242")}

def pcs(m, train_rows, k=16):
    """Principal components of an embedding, fitted on the training rows only; no embedding -> missing."""
    has_ = np.abs(m).sum(1) > 0
    t = m[train_rows & has_]
    mu = t.mean(0)
    _, _, vt = np.linalg.svd(t - mu, full_matrices=False)
    out = (m - mu) @ vt[:k].T
    out[~has_] = np.nan
    return out

def features(target, train_rows, analogue_est=None):
    """Sales: attributes + photo components; price: attributes + the landed cost (the declared feature sets)."""
    X = pd.DataFrame({c: pd.Categorical(items[c].astype("string")) for c in FC_CATEGORICAL})
    X["n_sizes"] = items["n_skus"].astype(float)
    X["net_weight"] = pd.to_numeric(items["NET_WEIGHT"], errors="coerce")
    if target == "sales":
        X["price"] = price_
        X["price_pct_in_cat"] = items.groupby("CAT_DES_EN")["PRICE_BASE_W_VAT"].rank(pct=True)
        for j, col in enumerate(pcs(blocks["image"], train_rows).T):
            X[f"img_pc{j + 1}"] = col
    else:
        X["log_cost"] = items["log_cost"]
    if analogue_est is not None:
        X["analogue"] = analogue_est
    return X

FAMILIES = [("img_pc", "photo"), ("analogue", "similar past products"), ("price", "price"), ("log_cost", "landed cost"),
            ("n_sizes", "number of sizes"), ("net_weight", "weight")]
family = lambda c: next((f for p, f in FAMILIES if c.startswith(p)), "attribute: " + c)

def spearman(a, b):
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    r = pd.Series(a).rank().corr(pd.Series(b).rank())
    return 0.0 if pd.isna(r) else float(r)

def boot_models(metric, rows, a, b, n=300):
    """95% interval of metric(b) - metric(a) on `rows`, resampling models (colourways of a model move together)."""
    rng = np.random.default_rng(0)
    idx_rows = np.flatnonzero(rows)
    uniq, inv = np.unique(GROUPS[idx_rows], return_inverse=True)
    members = [idx_rows[inv == g] for g in range(len(uniq))]
    d = [metric(b, ix) - metric(a, ix) for ix in
         (np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))]) for _ in range(n))]
    return np.percentile(d, [2.5, 97.5]).round(3)

### 4.1 Sales (whole-life units)

Compared on the same new products: the category median; the family + price band median; the **similar past products**
(similarity-weighted mean of the 10 most similar, other models only); and **LightGBM** on attributes, price and photo
components. Metrics: ranking within the category and among products launched in the same month; typical error factor;
share within a factor 2; precision on the top 20%.


In [ ]:
y = items["log_sales"].to_numpy(float)
month_group = (items["CAT_DES_EN"].astype(str) + "|" + items["launch"].dt.month.astype(str) + "|" + sea_).to_numpy()
cats = items["CAT_DES_EN"].to_numpy()

def weighted_spearman(yy, pp, keys_):
    res = [(spearman(yy[keys_ == g], pp[keys_ == g]), (keys_ == g).sum()) for g in np.unique(keys_) if (keys_ == g).sum() > 10]
    return np.average([r for r, _ in res], weights=[n for _, n in res]) if res else 0.0

def sales_metrics(p, rows, units=True):
    yy, pp, cc, gg = y[rows], p[rows], cats[rows], month_group[rows]
    top_true = pd.Series(yy).groupby(cc).rank(pct=True).to_numpy() > 0.8
    top_pred = pd.Series(pp + np.random.default_rng(0).random(len(pp)) * 1e-9).groupby(cc).rank(pct=True).to_numpy() > 0.8
    out = {"ranking within category": weighted_spearman(yy, pp, cc), "ranking, same launch month": weighted_spearman(yy, pp, gg)}
    if units:
        out |= {"error factor": np.exp(np.mean(np.abs(yy - pp))), "within 2x": np.mean(np.abs(yy - pp) < np.log(2)),
                "bias": np.exp(np.mean(pp - yy)) - 1}
    return out | {"top-20% precision": top_true[top_pred].mean()}

def median_by(cols, fit):
    d = items[cols].astype("string").assign(y=np.where(fit, y, np.nan))
    return d.groupby(cols, dropna=False)["y"].transform("median").fillna(d.groupby("CAT_DES_EN")["y"].transform("median")).to_numpy(float)

def analogue(target, w, pool_rows, k=10):
    """Similarity-weighted mean of the target over the k most similar products of the training seasons (other models)."""
    est = np.full(len(items), np.nan)
    for pool in CATEGORY_POOLS:
        cand = pool[pool_rows[pool] & np.isfinite(target[pool])]
        for start in range(0, len(pool), 1024):
            qs = pool[start:start + 1024]
            s = score_matrix(w, qs, cand).astype(float)
            s[not_allowed(qs, cand)] = -np.inf
            top = np.argsort(-s, axis=1, kind="stable")[:, :k]
            sc = np.take_along_axis(s, top, axis=1)
            ww = np.where(np.isfinite(sc), np.clip(sc, 0, None), 0.0)
            est[qs] = (ww * target[cand][top]).sum(1) / np.maximum(ww.sum(1), 1e-12)
    return est

items["price_band"] = pd.qcut(items["PRICE_BASE_W_VAT"].rank(method="first"), 5, labels=False).astype(str)

def sales_estimates(tr):
    fit = tr & np.isfinite(y)
    X_s = features("sales", tr)
    model_s = lgb.LGBMRegressor(**LGB).fit(X_s[fit], y[fit])
    ana = analogue(y, W_BASE, fit)
    est = {"category median": median_by(["CAT_DES_EN"], fit),
           "family + price band median": median_by(["CAT_DES_EN", "GFA_DES_EN", "price_band"], fit),
           "similar past products (engine)": np.where(np.isfinite(ana), ana, median_by(["CAT_DES_EN"], fit)),
           "gradient boosting": model_s.predict(X_s)}
    return est, model_s, X_s

for split_name, (tr, te) in SALES_SPLITS.items():
    est, sales_model, X_sales = sales_estimates(tr)
    ev_rows = te & np.isfinite(y)
    print(f"{split_name}: training {(tr & np.isfinite(y)).sum():,} | evaluated {ev_rows.sum():,} new products")
    print(pd.DataFrame({n: sales_metrics(p, ev_rows) for n, p in est.items()}).T.round(3).to_string(), "\n")
est_sales = est                                    # the test half, used below

gb, sim = est_sales["gradient boosting"], est_sales["similar past products (engine)"]
print("gradient boosting - similar past products, test half (95% CI, paired, by model):")
print("  same-month ranking", boot_models(lambda p, ix: weighted_spearman(y[ix], p[ix], month_group[ix]), ev_rows, sim, gb))
print("  log error saved   ", boot_models(lambda p, ix: -np.mean(np.abs(y[ix] - p[ix])), ev_rows, sim, gb))

tr242, te242 = RANK_242
est242, _, _ = sales_estimates(tr242)
print(f"\nranking check on 242 (232 + 241 -> 242, {(te242 & np.isfinite(y)).sum():,} products, sales cut: ranking only):")
print(pd.DataFrame({n: sales_metrics(p, te242 & np.isfinite(y), units=False) for n, p in est242.items()}).T.round(3).to_string())

tr, te = SALES_SPLITS["test (232 -> the other half)"]
fit = tr & np.isfinite(y)
shuffled = y.copy()
fit_idx = np.flatnonzero(fit)
shuffled[fit_idx] = shuffled[np.random.default_rng(0).permutation(fit_idx)]
p_shuffled = lgb.LGBMRegressor(**LGB).fit(X_sales[fit], shuffled[fit]).predict(X_sales)
print("\nleakage check, training target shuffled at random (ranking must fall to ~0):",
      {k: round(float(v), 3) for k, v in sales_metrics(p_shuffled, ev_rows).items()})

**Reading.** On the test half, LightGBM ranks same-month products at about 0.50, against 0.33 for the similar
products and 0.29 for family + price band; units are off by a factor of 1.7 (76% within ×2), against 2.0. On the same
products it beats the similar products on both (paired intervals +0.10 to +0.22 of ranking, +0.14 to +0.23 log units of
error saved); they stay as the explanation a buyer can check. On 242 the ranking holds (0.41 against 0.28). With a
shuffled target the ranking falls to zero: no leakage. Bias about −6%.


### 4.1b A range per product, and a second opinion with the buyer's plan

**Range.** LightGBM estimates the 10th and 90th percentiles of each product, widened by one constant so that 80% of the
validation half falls inside (*conformalised quantile regression*, Romano, Patterson & Candès, 2019); the test half
checks it.

**Second opinion.** The same model with `PUR_BET` and `TYPOLOGY`. More stock and more stores also cause sales, so it
checks a plan; it does not replace the estimate.


In [ ]:
def quantile_model(X, rows, alpha):
    return lgb.LGBMRegressor(**{**LGB, "objective": "quantile", "alpha": alpha}).fit(X[rows], y[rows]).predict(X)

r_tr, r_cal = SALES_SPLITS["validation (232 -> half of the new 241 models)"]
_, r_chk = SALES_SPLITS["test (232 -> the other half)"]
r_fit, r_cal, r_chk = r_tr & np.isfinite(y), r_cal & np.isfinite(y), r_chk & np.isfinite(y)
X_plain = features("sales", r_tr)
X_buyer = X_plain.assign(**{c: pd.Categorical(items[c].astype("string")) for c in ["PUR_BET", "TYPOLOGY"]})

range_rows, second = {}, {}
for variant, X_ in [("product only (delivered)", X_plain), ("+ buyer's plan", X_buyer)]:
    centre = lgb.LGBMRegressor(**LGB).fit(X_[r_fit], y[r_fit]).predict(X_)
    q_lo, q_hi = quantile_model(X_, r_fit, 0.1), quantile_model(X_, r_fit, 0.9)
    errs = np.maximum(q_lo - y, y - q_hi)[r_cal]
    widen = np.quantile(errs, min(1.0, 0.8 * (1 + 1 / len(errs))))      # from the validation half only
    resid = (y - centre)[r_cal]
    for kind, (lo_, hi_) in {"one factor for every product": (centre + np.quantile(resid, 0.1), centre + np.quantile(resid, 0.9)),
                             "quantile models, not calibrated": (q_lo, q_hi),
                             "conformalised quantiles": (np.minimum(q_lo - widen, centre), np.maximum(q_hi + widen, centre))}.items():
        inside = ((y >= lo_) & (y <= hi_))[r_chk]
        by_cat = pd.Series(inside).groupby(cats[r_chk]).mean()
        range_rows[(variant, kind)] = {"coverage": inside.mean(), "median width (high / low)": np.median(np.exp(hi_ - lo_)[r_chk]),
                                       "worst category": by_cat.min(), "best category": by_cat.max()}
    second[variant] = sales_metrics(centre, r_chk)
print(f"test half: {r_chk.sum():,} products (widening fitted on the {r_cal.sum():,} of the validation half)\n")
print(pd.DataFrame(range_rows).T.round(3).to_string(), "\n")
print(pd.DataFrame(second).T.round(3).to_string())

**Reading.** Uncalibrated, the quantiles cover two thirds of the test products; widened, 80%, with a narrower range
than one factor for all (×4.2 against ×5.0) and the same coverage in every category. The price keeps one factor (×0.89
to ×1.23). With the buyer's plan the same-month ranking rises from 0.50 to 0.60 and the top-20% precision from a third to
a half.


**Explaining a sales estimate: SHAP.** TreeSHAP gives the exact contribution of each feature for every product,
grouped into business families (a single photo component means nothing to a buyer).


In [ ]:
def shap_families(model, X, rows):
    sv = pd.DataFrame(model.predict(X[rows], pred_contrib=True), columns=list(X.columns) + ["expected"]).drop(columns="expected")
    return sv.T.groupby(lambda c: family(c)).sum().T

fam_sales = shap_families(sales_model, X_sales, ev_rows)
print("what drives the sales estimates (mean |SHAP|, log units, test products):")
print(fam_sales.abs().mean().sort_values(ascending=False).head(10).round(3).to_string())
k_ex = items[KEY].to_numpy()[ev_rows][7]
i_ex = int(np.flatnonzero(items[KEY].to_numpy() == k_ex)[0])
contrib_ex = shap_families(sales_model, X_sales, np.arange(len(items)) == i_ex).iloc[0]
print(f"\n{k_ex} ({items['GFS_DES_EN'].iat[i_ex]}, {items['CLR_DES'].iat[i_ex]}): sold {int(qty.iat[i_ex]):,}, "
      f"estimate {np.expm1(est_sales['gradient boosting'][i_ex]):,.0f} units")
print(contrib_ex.reindex(contrib_ex.abs().sort_values(ascending=False).index).head(5).round(2).to_string())

### 4.2 Price (landed cost × markup)

The model learns the markup, log(price) − log(cost), from the attributes and the cost; a product without a cost uses a
model without it. It never sees the price. The similar products enter as a feature only if their validation gain
excludes 0. Business check: the estimate rounded to the nearest of the 40 list prices is the right one or its
neighbour.


In [ ]:
yp = np.log(price_.to_numpy(float))
lc = items["log_cost"].to_numpy(float)
has_cost = np.isfinite(lc)
points = np.sort(price_.unique())
nearest = lambda v: np.abs(v[:, None] - points[None, :]).argmin(1)

def price_metrics(p, rows):
    ye, pe = np.exp(yp[rows]), np.exp(p[rows])
    steps = np.abs(nearest(pe) - nearest(ye))
    return {"mean error €": np.mean(np.abs(ye - pe)), "bias": np.exp(np.mean(p[rows] - yp[rows])) - 1,
            "exact list price": (steps == 0).mean(), "right or next": (steps <= 1).mean()}

def median_price(target, cols, fit):
    d = items[cols].astype("string").assign(y=np.where(fit, target, np.nan))
    return d.groupby(cols, dropna=False)["y"].transform("median").fillna(d.groupby("CAT_DES_EN")["y"].transform("median")).to_numpy(float)

SUB = ["CAT_DES_EN", "GFA_DES_EN", "GFS_DES_EN"]
def price_estimates(tr):
    ana = analogue(yp, {**W_BASE, "numeric": 0.0}, tr)          # the numeric block contains the price: never used
    sub_med = median_price(yp, SUB, tr)
    X_nc = features("price", tr).drop(columns="log_cost")
    no_cost = lgb.LGBMRegressor(**LGB).fit(X_nc[tr], yp[tr]).predict(X_nc)
    est = {"category median": median_price(yp, ["CAT_DES_EN"], tr), "sub-family median": sub_med,
           "cost × sub-family median markup": np.where(has_cost, lc + median_price(yp - lc, SUB, tr & has_cost), sub_med),
           "similar past products (engine)": np.where(np.isfinite(ana), ana, sub_med),
           "gradient boosting without cost": no_cost}
    models = {}
    for name, a in [("gradient boosting (cost × markup)", None), ("gradient boosting (cost × markup) + similar products", ana)]:
        X_p = features("price", tr, a)
        models[name] = (lgb.LGBMRegressor(**LGB).fit(X_p[tr & has_cost], (yp - lc)[tr & has_cost]), X_p)
        est[name] = np.where(has_cost, lc + models[name][0].predict(X_p), no_cost)
    return est, models

mae = lambda p, ix: -np.mean(np.abs(np.exp(yp[ix]) - np.exp(p[ix])))
for split_name, (tr, te) in PRICE_SPLITS.items():
    est_p, price_models = price_estimates(tr)
    print(f"{split_name}: training {tr.sum():,} | evaluated {te.sum():,} new products")
    print(pd.DataFrame({n: price_metrics(p, te) for n, p in est_p.items()}).T.round(3).to_string())
    gain = boot_models(mae, te, est_p["gradient boosting (cost × markup)"],
                     est_p["gradient boosting (cost × markup) + similar products"])
    if split_name.startswith("validation"):
        use_ana = gain[0] > 0                                   # the rule, decided on validation only
        print(f"similar products as a feature, validation gain {gain} € -> {'kept' if use_ana else 'not kept'}\n")
chosen = "gradient boosting (cost × markup)" + (" + similar products" if use_ana else "")
print(f"\nchosen ({chosen}) - similar past products, test (95% CI, paired, € of error saved):",
      boot_models(mae, te, est_p["similar past products (engine)"], est_p[chosen]))
price_model, X_price = price_models[chosen]
print("\nwhat drives the markup (mean |SHAP|, log units, test products with a cost):")
print(shap_families(price_model, X_price, te & has_cost).abs().mean().sort_values(ascending=False).head(6).round(3).to_string())

On the new 242 products the mean error is about 2.7 € (3.4 € without the cost, 3.8 € for the similar products, 4.2 €
for the sub-family median), and the right list price or its neighbour is found for 78%. Against the similar products the
model saves 0.9 to 1.3 € per product. The remaining bias (−4.5%) is a rise of markups in 242 that past seasons do not
announce; three corrections were tried, none gains.

Delivered estimates are **never fitted on the product itself** (5 folds grouped by model where the target is known).
Each carries an 80% range and, for the sales, the second opinion. **Stock for a sales period is not estimated**: the
data has whole-life units only, no dates and no stores, and the planned life cycle (`ITEM_LIFE_CYCLE_WEEK`, 4–12 weeks)
is not the selling time (the 241 products in the outlet stayed 46 weeks at full price).


### 4.3 Better models, a better metric, and problems before the prediction

CatBoost, the weeks on sale, a metric besides the median, problems in the data, the landed cost and the photos. These
were run separately with the same data, splits and features as sections 4.1 and 4.2; the saved results are loaded.


**CatBoost against LightGBM** (sales: train 232, test half of 241; price: 232 + 241 → 242).


In [ ]:
OUT5 = ROOT / "outputs" / "phase5b"
cb = pd.read_csv(OUT5 / "catboost_results.csv", encoding="utf-8-sig")
keep = ["target", "split", "model", "n_eval", "spearman_same_month", "mae_log", "within_2x", "mae_eur", "within_1_point"]
cb[cb["model"].str.contains("LightGBM|CatBoost|average", regex=True)][keep].round(3).reset_index(drop=True)

**Reading.** For sales CatBoost is better on this split (ranking +0.017 to +0.075) but not out of fold (below). For
price it is not better.


**Weeks on sale and the average, out of fold.** Season 241 had 15 to 53 weeks on sale at the extraction (232: 39 to
83). Trained on 232 alone the weeks hurt; once 241 is in the training they help.


In [ ]:
ex = pd.read_csv(OUT5 / "exposure_results.csv", encoding="utf-8-sig")
oof = ex[ex["split"].str.startswith("out of fold, 232 + 241")].drop(columns="split").set_index("model")
oof[["n_eval", "same-month ranking", "mae_log", "wape", "bias_log"]].round(3)

**Reading.** Out of fold, the weeks on sale improve LightGBM's ranking (0.625 → 0.660), CatBoost alone is worse, and
the **average** of the two ranks best (0.673) with the same error. That is the delivered sales model.


**A metric besides the median.** The median stays as the baseline. Headline metric: the **WAPE** (units missed over
units sold): it weights each product by its sales and stays defined for tiny sellers. The MAPE is unusable (it explodes
on products that sell very little).


In [ ]:
mt = pd.read_csv(OUT5 / "metrics_results.csv", encoding="utf-8-sig")
head = mt[(mt["section"] != "aggregation") & mt["metric"].isin(["wape", "mape", "rmsle", "within_2x"]) & (mt["target"] == "sales")
          & (mt["split"] == "test")]
head.pivot_table(index="model", columns="metric", values="value", aggfunc="first").round(3)

**Problems before the prediction.** The two serious ones: the sales are what the order and the allocation allowed,
not demand (the planning classes alone explain 19–24% of the variance), and season 241 is not a whole life. The first
cannot be corrected with this data.


In [ ]:
ua = pd.read_csv(OUT5 / "upstream_audit.csv", encoding="utf-8-sig")
ua[ua["severity"].isin(["high", "medium"])][["rank", "area", "status", "severity", "finding"]].reset_index(drop=True)

**The landed cost.** A wrong cost goes straight into the price. 38 costs are almost certainly wrong (not positive,
above the price, or more than 5 times away from the other colours of the model) and 248 are suspect. Ignoring the 38
gives no gain; ignoring the suspect ones helps on validation but not on the test, so they are only flagged.


In [ ]:
cc = pd.read_csv(OUT5 / "cost_check_results.csv", encoding="utf-8-sig")
cc[cc["model"] == "gradient boosting, no analogues"][["split", "variant", "n_eval", "mae_eur", "mape", "within_1_point"]].round(3)

**Palette and photos.** The text does not count the colour twice (correlation −0.01). The palette is weak when the
photo shows another item or is not a packshot (65–70% of the dominant colours fit the colour name, against 74%); those
292 photos no longer feed the palette block. 317 photos show a person and 720 several pieces; no new crop is applied
without a review.


In [ ]:
px = pd.read_csv(ROOT / "data" / "embeddings" / "palette_excluded.csv", encoding="utf-8-sig")
px["reason"].value_counts().to_frame("photos out of the palette")

### 4.4 The delivered sales estimate

The **mean of LightGBM and CatBoost**, trained with the weeks on sale and made at 64 weeks (the median of 232): "units
over a standard whole life" for every product. Out of fold on 241: typical error ×1.49, 86% within ×2, range coverage
80% (77% on 232). Not comparable with the ×1.7 of section 4.1, which came from one split trained on 232 only.


### 4.5 Where the price estimate misses: the largest errors

The mean error hides where the model fails. The error is shown in euros **and in percent of the price**, by category:
4 € is small on a 60 € coat and large on 6 € earrings. Official test 232 + 241 → 242, and the validation split 232 → 241
for comparison. "Worst" = the 10% largest errors.


In [ ]:
perr = pd.read_csv(OUT5 / "price_errors.csv", encoding="utf-8-sig")
print(f"official test: {len(perr):,} new 242 products, mean error {perr['abs_error_eur'].mean():.2f} €, "
      f"{perr['abs_error_pct'].mean():.1%} of the price; the 10% largest errors in € carry "
      f"{perr.loc[perr['worst_eur'], 'abs_error_eur'].sum() / perr['abs_error_eur'].sum():.0%} of all the euros of error, "
      f"and {(perr.loc[perr['worst_eur'], 'direction'] == 'too low').mean():.0%} of them are estimates too low")


def pe_by(col, min_n=1):
    g = perr.groupby(col)
    t = pd.DataFrame({"products": g.size(),
                      "mean error €": g["abs_error_eur"].mean().round(2),
                      "mean error % of price": (100 * g["abs_error_pct"].mean()).round(1),
                      "error >= 4 € (%)": (100 * perr["abs_error_eur"].ge(4).groupby(perr[col]).mean()).round(0),
                      "share of the test (%)": (100 * g.size() / len(perr)).round(0),
                      "share of the worst in € (%)": (100 * g["worst_eur"].sum() / perr["worst_eur"].sum()).round(0),
                      "share of the worst in % (%)": (100 * g["worst_pct"].sum() / perr["worst_pct"].sum()).round(0)})
    return t[t["products"] >= min_n].sort_values("mean error €", ascending=False)


pe_by("CAT_DES_EN")

In [ ]:
print("families with the largest mean error (20 products or more):")
print(pe_by("GFA_DES_EN", 20).head(6).to_string())
print("\ncollection type (20 products or more):")
print(pe_by("COLLECTION_TYPE", 20).to_string())
print("\nwith and without a landed cost:")
print(pe_by("cost_status").to_string())

**Reading.** The 10% largest errors carry half of all the euros of error, and three quarters are **too low**. In
euros the model fails on **apparel** (25% of the test, 61% of the worst; a third miss by 4 € or more); jewellery and
wallets miss by 1.3–1.4 €. In percent it is **jewellery** (39% of the test, 54% of the worst), because its prices are
low. Worst families: **coats** (78% miss by 4 € or more, 17 € on average), raincoats, dresses. **Exclusive Online** is 8%
of the test and a quarter of the worst; one bridal capsule (17 products, up to 250 €, cost recorded as 25.00 €) carries
13% of all the error. Products **without a cost** (3%) err three times more (7.7 € against 2.5 €).


In [ ]:
ve_price = pd.read_csv(OUT5 / "validation_errors.csv", encoding="utf-8-sig").query("target == 'price'")
print("share of the products and of the worst errors in €, by category (%): validation 241 against the test 242")
print(pd.DataFrame({
    "241: products": ve_price["CAT_DES_EN"].value_counts(normalize=True),
    "241: worst in €": ve_price.loc[ve_price["worst_eur"], "CAT_DES_EN"].value_counts(normalize=True),
    "242: products": perr["CAT_DES_EN"].value_counts(normalize=True),
    "242: worst in €": perr.loc[perr["worst_eur"], "CAT_DES_EN"].value_counts(normalize=True)}).mul(100).round(0).to_string())

nocost = pd.read_csv(OUT5 / "nocost_rule.csv", encoding="utf-8-sig").set_index("split")
print("\nproducts without a cost: fallback model against the similar products (95% interval, + = similar products better):")
print(nocost[["n_no_cost", "fallback_mae_eur", "analogue_mae_eur", "ci_eur_low", "ci_eur_high"]].round(2).to_string())

lcurve = pd.read_csv(OUT5 / "learning_curve.csv", encoding="utf-8-sig").query("target == 'price'")
print("\nlearning curve of the price model (validation 232 -> 241, mean of the draws):")
print(lcurve.groupby("fraction")[["training models", "train MAE €", "validation MAE €"]].mean().round(3).to_string())

**Reading.** The same groups fail on 241, so the pattern belongs to the model and the data, not to one season; in 242
the errors lean low because the markups rose.

**Products without a cost: not changed.** On the test the similar products would do better (+0.7 to +2.7 € per
product), but that was seen on the test. Decided on validation with a rule fixed beforehand: 241 has only 6 such
products and the rule does not pass, so the test stays a blind check.

**More data would not help the price much**: the validation error is flat from half of the training data (2.37 € →
2.29 €). Products with the same attributes and cost still differ by 12.6% in price: part of the price is not in the
data. What would help: correct costs and a range per category (the 80% range covers 91% of hand bags, 64% of Exclusive
Online, 72% of jewellery).


## 5. Improvements tried against the baseline

An idea is kept only if it wins on signals it was not tuned on and no guard falls by more than 1 point.

### 5.1 Fusion: standardise each block's similarity per category — rejected

Photo similarities are compressed (random pairs about 0.8, best matches 0.93) while the categorical one spans 0–1.
Idea: standardise each block against random pairs of the same category before the weighted sum.


In [ ]:
blocks["image"] = clip_m      # the experiments of this section were run against the baseline engine
rng = np.random.default_rng(0)
stats = {}
for pool in CATEGORY_POOLS:
    cat_name = items["CAT_DES_EN"].iat[pool[0]]
    a, b = rng.choice(pool, 20000), rng.choice(pool, 20000)
    keep = items["PROD_REF"].to_numpy()[a] != items["PROD_REF"].to_numpy()[b]
    a, b = a[keep], b[keep]
    for bl in BLOCKS:
        both = (has[bl][a] > 0) & (has[bl][b] > 0)
        v = np.einsum("ij,ij->i", blocks[bl][a[both]], blocks[bl][b[both]])
        stats[(cat_name, bl)] = (v.mean(), v.std())
print(pd.DataFrame({bl: [round(stats[(c, bl)][0], 2) for c in sorted(items["CAT_DES_EN"].unique())] for bl in BLOCKS},
                   index=sorted(items["CAT_DES_EN"].unique())).to_string(), "\n(mean similarity of random pairs)")
pair_q = np.zeros(len(S), int)
pair_c = np.zeros(len(S), int)
pos_ = pd.Series(np.arange(len(items)), index=items[KEY])
for (qk, ck), r_ in pair_row.items():
    pair_q[r_], pair_c[r_] = pos_[qk], pos_[ck]
qcat = items["CAT_DES_EN"].to_numpy()[pair_q]
Z = S.copy()
for k_, bl in enumerate(BLOCKS):
    mu = np.array([stats[(c, bl)][0] for c in qcat])
    sd = np.array([stats[(c, bl)][1] for c in qcat])
    Z[:, k_] = np.where(H[:, k_] > 0, (S[:, k_] - mu) / sd, 0)
z_fixed, z_cv = team_score(Z)
pd.DataFrame({"fixed weights": [acc(base_fixed), acc(z_fixed)], "5-fold CV refit": [acc(base_cv), acc(z_cv)],
              "CV gain, 95% CI": ["", "{:+.3f} to {:+.3f}".format(*boot_diff(base_cv, z_cv))]},
             index=["raw cosine (baseline)", "standardised per category"]).round(3)

Worse in cross-validation (the interval excludes 0): the team-fitted weights already compensate the compressed photo
scale. **Not adopted.**

### 5.2 Less weight on colour and theme — rejected

The brief's four answers for `167718_BU` have other colours and another theme and rank far down. Hypothesis: colour
(weight 2) and theme (1.5) weigh too much.


In [ ]:
cat_base = cat_m.copy()
BRIEF = ["169597_FU", "169607_IG", "169607_PE", "177781_LB"]
brief_idx = [int(pos_[k]) for k in BRIEF]

def brief_ranks(w):
    s = score_matrix(w, np.array([q]), next(p for p in CATEGORY_POOLS if q in p))[0].astype(float)
    pool = next(p for p in CATEGORY_POOLS if q in p)
    s[not_allowed(np.array([q]), pool)[0]] = -np.inf
    return [int((s > s[pool == j][0]).sum()) + 1 for j in brief_idx]

rows = {}
for name, factors in [("baseline", {}), ("colour ×0.5", {"CLR_DES": 0.5}), ("colour 0", {"CLR_DES": 0}),
                      ("theme 0", {"THEME": 0}), ("colour 0 + theme 0", {"CLR_DES": 0, "THEME": 0})]:
    m = cat_base.copy()
    for a, f in factors.items():
        s0_, e0_ = cat_attrs[a]
        m[:, s0_:e0_] *= np.sqrt(f)
    blocks["categorical"] = normalise_rows(m)
    S2 = S.copy()
    S2[:, BLOCKS.index("categorical")] = np.einsum("ij,ij->i", blocks["categorical"][pair_q], blocks["categorical"][pair_c])
    fx, cv = team_score(S2)
    g = evaluate(top_k(W_BASE, TEST_Q, k=4))
    rows[name] = {"team, fixed": acc(fx), "team, CV": acc(cv), "brief ranks": brief_ranks(W_BASE),
                  "same sub-family": g["same_subfamily"], "same material": g["same_material"], "sales corr.": g["sales_corr"]}
blocks["categorical"] = cat_base
pd.DataFrame(rows).T

The team's agreement falls when colour loses weight, the brief's products move further down, and theme 0 costs 4
points of sales correlation. **Not adopted.**

### 5.3 Image encoder — FashionSigLIP adopted, DINOv2 rejected

The photo is the block the team weighs most. Three encoders, run once on the same crops, against the Sprint 2 CLIP:


In [ ]:
enc = pd.read_csv(ROOT / "outputs" / "phase2b" / "image_encoders.csv", index_col="encoder")
enc[["team, fixed weights", "team, 5-fold CV refit", "photo alone: same sub-family", "photo alone: same material",
     "photo alone: same finish", "photo alone: same colour", "brief ranks (photo alone)", "hit@4", "sales_corr"]].round(3)

### 5.4 Diversity: at most 1 or 2 colours of one model in the top 5 — rejected

Re-ranking the top 50 so that a model appears at most once or twice:


In [ ]:
top50 = top_k(W_BASE, k=50)
ref_arr = items["PROD_REF"].astype(str).to_numpy()
judged = {}
for a in answers:
    for c in a["similar"]:
        judged.setdefault((a["query"], c), []).append(1.0)
    for c in a["not"]:
        judged.setdefault((a["query"], c), []).append(0.0)

def cap(nb, m):
    if m is None:
        return nb[nb["rank"] <= 10]
    nb = nb.assign(model=ref_arr[nb["c"]]).sort_values(["q", "rank"])
    nb = nb[nb.groupby(["q", "model"]).cumcount() < m].copy()
    nb["rank"] = nb.groupby("q").cumcount() + 1
    return nb[nb["rank"] <= 10]

rows = {}
for name, m in [("baseline (no cap)", None), ("at most 2 colours per model", 2), ("at most 1 colour per model", 1)]:
    nb = cap(top50, m)
    t5 = nb[nb["rank"] <= 5]
    g = evaluate(nb[nb["q"].isin(TEST_Q)])
    keys_ = items[KEY].to_numpy()
    j = [np.mean(judged[(keys_[a], keys_[b])]) for a, b in zip(t5["q"], t5["c"]) if (keys_[a], keys_[b]) in judged]
    rows[name] = {"distinct models in top 5": t5.assign(m=ref_arr[t5["c"]]).groupby("q")["m"].nunique().mean(),
                  "same sub-family": g["same_subfamily"], "same material": g["same_material"],
                  "team: judged top-5 called similar": np.mean(j)}
pd.DataFrame(rows).T.round(3)

The baseline already shows 4.6 different models in the top 5: the problem is rare. One colour per model costs 2.4
points of material agreement; two change almost nothing. **Not adopted.**


### 5.5 Two-stage retrieval: the nearest photos first, then the full score — not adopted

The N nearest photos (within the sub-family or the whole category, with or without one colour per model) are re-ranked
by the full score and put first. Run once against the Sprint 3 engine, with the whole catalogue ranked for the team:


In [ ]:
ts = pd.read_csv(ROOT / "outputs" / "phase4" / "twostage_check.csv", index_col=0)
shown = {"baseline (current engine)": "Sprint 3 engine",
         "cap 1 colour per model": "cap 1 colour per model",
         "two-stage 50, re-rank with the full score": "two-stage 50, sub-family",
         "two-stage 50 in the category, full score (PM)": "two-stage 50, category",
         "two-stage 50 in the category, full score + cap (PM default)": "two-stage 50, category + cap"}
t = ts.loc[list(shown)].rename(index=shown)
pts = lambda c: (100 * t[c]).round(1)
pd.DataFrame({"team pairs right": pts("team pairs right"),
              "team diff, 95% CI": [("" if pd.isna(lo) else f"{100 * lo:+.1f} to {100 * hi:+.1f}")
                                    for lo, hi in zip(t["team diff low"], t["team diff high"])],
              "hit@4": pts("hit@4"), "same sub-family": pts("same_subfamily"), "same material": pts("same_material"),
              "same finish": pts("same_finishing"), "sales corr.": t["sales_corr"].round(3),
              "top 5 judged": pts("top 5 judged by the team"), "brief ranks": t["brief ranks"]})

Only the two-stage over the **whole category, without the cap** is preferred by the team (+2.2 points, the interval
excludes 0) and brings three of the brief's answers far forward. But hit@4 falls 8.5 points, material 8.1, sub-family
3.7, well beyond the guard; nine variants were tried and the team judged candidates of the current engine. **Not
adopted.**

## 6. Photos of a person wearing the item

A photo cropped to its non-white pixels becomes the whole person when a model wears the item (a swimsuit embedded as
white trousers). Run once (about 17 hours on a CPU): CLIP on the whole photo as a cheap person filter, then the
**OWLv2** detector on the 2,454 photos above it, for a person score and the box of the item:


In [ ]:
pc = pd.read_parquet(EMB_DIR / "person_crops.parquet")
masks = pd.read_parquet(EMB_DIR / "masks.parquet").set_index(KEY)
pc["phase1b_crop"] = masks.loc[pc[KEY], "mask_method"].to_numpy()
person = pc[pc["person_score"] >= 0.10]
print(f"checked by OWLv2: {len(pc):,} | with a person (score >= 0.10): {len(person)}")
print(person.groupby("phase1b_crop").size().rename("photos").to_string())
print(f"\nautomatic re-crop rule: {int(pc['auto_rule'].sum())} photos | confirmed by eye: {int(pc['recrop'].sum())}:",
      ", ".join(pc.loc[pc["recrop"], KEY]))

**Most photos of a person were already fine** (close-ups had been cropped to the item in Sprint 1). The automatic
rule picks 13 photos: 6 better, 7 worse (printed faces, zooms, sets cut to one piece). **Only the 6 confirmed by eye
are re-cropped.**


In [ ]:
plt.figure(figsize=(14, 8))
plt.imshow(Image.open(ROOT / "outputs" / "phase2b" / "person_recrops_checked.jpg"))
plt.axis("off")
plt.title("The 13 automatic re-crops, before | after (kept: 211451_BM, 221878_BK, 221878_WT, 222924_BK, 223997_NM, 224206_IG)")
plt.show()

## 7. The Sprint 3 engine against the baseline

Same weights as Sprint 2; only the photo vectors change (FashionSigLIP, with the 6 re-crops).


In [ ]:
sig_meta = json.loads((EMB_DIR / "image_fashionsiglip.json").read_text(encoding="utf-8"))
sig = np.load(EMB_DIR / "image_fashionsiglip.npy").astype(np.float32)
idx = items[KEY].map(pd.Series(range(len(sig_meta["keys"])), index=sig_meta["keys"]))
sig_m = np.zeros((len(items), sig.shape[1]), np.float32)
ok = idx.notna().to_numpy() & items["img_trusted"].to_numpy()
sig_m[ok] = sig[idx[ok].astype(int).to_numpy()]
sig_m = normalise_rows(sig_m)
clip_m = blocks["image"]

def photo_alone(m):
    """Share of each photo's 4 nearest photos (photo only) that share a label the photo never saw (test half)."""
    qs_all, cs_all = [], []
    for pool in CATEGORY_POOLS:
        pool_p = pool[np.abs(m[pool]).sum(1) > 0]
        qs = pool_p[np.isin(pool_p, TEST_Q)]
        for st in range(0, len(qs), 1024):
            qq = qs[st:st + 1024]
            s = (m[qq] @ m[pool_p].T).astype(float)
            s[not_allowed(qq, pool_p)] = -np.inf
            top = pool_p[np.argsort(-s, axis=1, kind="stable")[:, :4]]
            qs_all.append(np.repeat(qq, 4))
            cs_all.append(top.ravel())
    qi, ci = np.concatenate(qs_all), np.concatenate(cs_all)
    out = {}
    for name, col in [("sub-family", "GFS_DES_EN"), ("material", "MATERIAL"), ("finish", "FINISHING"), ("colour", "CLR_DES")]:
        v = items[col].astype("string").to_numpy(object, na_value=None)
        both = pd.notna(v[qi]) & pd.notna(v[ci])
        out[f"photo alone: same {name}"] = (v[qi][both] == v[ci][both]).mean()
    return out

rows, team = {}, {}
for name, m in [("baseline (CLIP)", clip_m), ("Sprint 3 (FashionSigLIP)", sig_m)]:
    blocks["image"] = m
    has["image"] = (np.abs(m).sum(1) > 0).astype(np.float32)
    S3 = S.copy()
    k_img = BLOCKS.index("image")
    S3[:, k_img] = np.where(H[:, k_img] > 0, np.einsum("ij,ij->i", m[pair_q], m[pair_c]), 0)
    team[name] = team_score(S3)
    g = evaluate(top_k(W_BASE, TEST_Q, k=4))
    rows[name] = {"team, fixed weights": acc(team[name][0]), "team, CV refit": acc(team[name][1]), **photo_alone(m),
                  "hit@4": g["hit@4"], "same sub-family": g["same_subfamily"], "same material": g["same_material"],
                  "sales corr.": g["sales_corr"], "brief ranks": brief_ranks(W_BASE)}
lo, hi = boot_diff(team["baseline (CLIP)"][1], team["Sprint 3 (FashionSigLIP)"][1])
print(f"team preferences, CV: difference "
      f"{acc(team['Sprint 3 (FashionSigLIP)'][1]) - acc(team['baseline (CLIP)'][1]):+.3f}, 95% CI {lo:+.3f} to {hi:+.3f}")
pd.DataFrame(rows).T

The Sprint 3 engine sees more in the photos (+10 points of sub-family and +9 of material from the photo alone), keeps
every guard within 1 point and moves the brief's answers up. Its team gain (+1 to +1.5 points) is within the noise of
99 queries, and the encoder was chosen on the same metric.

## Summary

| | Result |
|---|---|
| Baseline | Sprint 2 engine rebuilt: `hit@4` 91.5%, team preferences 72.8% in cross-validation |
| Explanations | Exact decomposition of every match (block shares, shared attributes and materials, differences, close colours), complete to 1e-6; the photo best separates the team's judgements |
| Price | Every product has a price. Cost × estimated markup, test on 242: mean error ≈ 2.7 € (similar products 3.8 €), right price or its neighbour 78%, bias −4.5%. Largest errors: expensive apparel (coats), Exclusive Online, products without a cost; in % cheap jewellery |
| Sales | Whole-life units, train 232, test 241. Same-month ranking 0.50 (similar products 0.33), ×1.7 (76% within ×2), no leakage; 80% range per product; second opinion with the buyer's plan (ranking 0.60). Delivered: LightGBM + CatBoost with the weeks on sale (×1.49 out of fold). No stock per period (no dates or stores) |
| Kept | FashionSigLIP image encoder; 6 photos re-cropped after a check by eye |
| Rejected | Standardised fusion, lower colour/theme weights, DINOv2, diversity cap, blind re-cropping, two-stage retrieval (team +2.2 points, but hit@4 −8.5 and material −8.1) |
